In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path
from collections import Counter

root = Path('/kaggle/input')

print('ATTACHED DATASETS')
for d in sorted(root.iterdir()):
    print('  ', d.name)

print('\nFILE TYPES')
ext = Counter(p.suffix.lower() for p in root.rglob('*') if p.is_file())
for e, n in ext.most_common(12):
    print(f'   {e or "(none)":8s} {n:6d}')

print('\nTIFF FIELDS')
tifs = [p for p in root.rglob('*') if p.suffix.lower() in {'.tif', '.tiff'}]
print(f'   {len(tifs)} found')
for p in tifs[:10]:
    print('   ', p)

print('\nPOSSIBLE GROUND-TRUTH MASKS')
for pat in ['*mask*', '*gt*', '*sam*', '*.npy', '*.npz']:
    hits = [h for h in root.rglob(pat)][:5]
    if hits:
        print(f'   {pat}')
        for h in hits:
            print('     ', h)

In [ ]:
from pathlib import Path
from collections import Counter

root = Path('/kaggle/input')

print('ATTACHED DATASETS')
for d in sorted(root.iterdir()):
    n = sum(1 for _ in d.rglob('*') if _.is_file())
    mb = sum(f.stat().st_size for f in d.rglob('*') if f.is_file()) / 1e6
    print(f'   {d.name:40s} {n:7,d} files  {mb:9.1f} MB')

print('\nFILE TYPES')
for e, n in Counter(p.suffix.lower() for p in root.rglob('*')
                    if p.is_file()).most_common(10):
    print(f'   {e or "(none)":8s} {n:7,d}')

print('\nTOP-LEVEL STRUCTURE (2 levels)')
for d in sorted(root.iterdir()):
    for sub in sorted(d.iterdir())[:6]:
        k = sum(1 for _ in sub.rglob('*') if _.is_file()) if sub.is_dir() else 0
        print(f'   {sub.relative_to(root)}{"/" if sub.is_dir() else ""}  '
              f'{k if sub.is_dir() else ""}')
    if len(list(d.iterdir())) > 6:
        print(f'   ... {len(list(d.iterdir()))} entries total')

imgs = [p for p in root.rglob('*')
        if p.suffix.lower() in {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}]
print(f'\nIMAGE FILES: {len(imgs):,}')
for p in imgs[:12]:
    print('   ', p.relative_to(root))

if imgs:
    import cv2
    print('\nSAMPLE IMAGE PROPERTIES')
    for p in imgs[:3]:
        im = cv2.imread(str(p))
        print(f'   {p.name:30s} {im.shape if im is not None else "UNREADABLE"}  '
              f'{p.stat().st_size/1e6:.1f} MB')

print('\nSLIDE FOLDERS (candidate batch identifiers)')
folders = sorted({p.parent.name for p in imgs})
print(f'   {len(folders)} distinct')
for f in folders[:20]:
    print('   ', f)

In [ ]:
from pathlib import Path
root = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd')

print('DIRECTORY TREE')
for p in sorted(root.rglob('*')):
    if p.is_dir():
        n = sum(1 for _ in p.glob('*.tiff'))
        print(f'   {p.relative_to(root)}/   {n} tiff' if n else f'   {p.relative_to(root)}/')

print('\nNON-IMAGE FILES (labels, metadata)')
others = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() != '.tiff']
print(f'   {len(others)} found')
for p in others[:20]:
    print('   ', p.relative_to(root), p.stat().st_size, 'bytes')

print('\nFIELDS PER SLIDE')
import collections
c = collections.Counter(p.parent.name for p in root.rglob('*.tiff'))
print(f'   {len(c)} slides, {sum(c.values())} fields, '
      f'{min(c.values())}-{max(c.values())} per slide')

print('\nDATES PRESENT')
print('  ', sorted({s[:6] for s in c}))

In [ ]:
from pathlib import Path
root = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd')

print('DIRECTORY TREE')
for p in sorted(root.rglob('*')):
    if p.is_dir():
        n = sum(1 for _ in p.glob('*.tiff'))
        print(f'   {p.relative_to(root)}/   {n} tiff' if n else f'   {p.relative_to(root)}/')

print('\nNON-IMAGE FILES (labels, metadata)')
others = [p for p in root.rglob('*') if p.is_file() and p.suffix.lower() != '.tiff']
print(f'   {len(others)} found')
for p in others[:20]:
    print('   ', p.relative_to(root), p.stat().st_size, 'bytes')

print('\nFIELDS PER SLIDE')
import collections
c = collections.Counter(p.parent.name for p in root.rglob('*.tiff'))
print(f'   {len(c)} slides, {sum(c.values())} fields, '
      f'{min(c.values())}-{max(c.values())} per slide')

print('\nDATES PRESENT')
print('  ', sorted({s[:6] for s in c}))

In [ ]:
from pathlib import Path
import numpy as np, cv2, pandas as pd

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/'
            'thin_films_part1/sickle3-edofed')

# one field per slide, the middle by filename sort - your original rule
fields = []
for d in sorted(ROOT.iterdir()):
    if d.is_dir():
        t = sorted(d.glob('*.tiff'))
        if t:
            fields.append(t[len(t)//2])
print(f'{len(fields)} fields, one per slide\n')

rows = []
for f in fields:
    img = cv2.imread(str(f))
    g = cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, bw = cv2.threshold(g, 127, 255,
                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8), 2)
    n, lab, st, _ = cv2.connectedComponentsWithStats(bw, 8)
    areas = st[1:, cv2.CC_STAT_AREA]
    areas = areas[areas > 3000]
    med = np.median(areas) if len(areas) else 0
    rows.append(dict(slide=f.parent.name, field=f.stem,
                     blobs=len(areas), median_area=med,
                     est_cells=int(np.round(areas / max(med, 1)).sum()),
                     foreground_pct=100 * (bw > 0).mean()))
    print(f'  {f.parent.name}  {len(areas):3d} blobs, median {med:6.0f} px')

df = pd.DataFrame(rows)
df.to_csv('/kaggle/working/field_survey.csv', index=False)
print('\n', df[['blobs', 'median_area', 'est_cells',
                'foreground_pct']].describe().to_string())
print(f'\nsingle-cell area estimate: {df.median_area.median():.0f} px')
print('paper reports ~10,300 px for a 7.5 um erythrocyte at this scale')

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth \
      -O /kaggle/working/sam_vit_b.pth
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
from pathlib import Path
p = Path('/kaggle/working/sam_vit_b.pth')
print('weights present :', p.exists())
print('size            :', f'{p.stat().st_size/1e6:.0f} MB' if p.exists() else '-')
print('  (should be ~375 MB)')

try:
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    print('segment_anything : OK')
except Exception as e:
    print('segment_anything : FAILED —', e)

import torch
print('GPU              :', torch.cuda.is_available())

In [ ]:
!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth \
      -O /kaggle/working/sam_vit_b.pth

In [ ]:
import numpy as np, cv2, torch, time
from pathlib import Path
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/'
            'thin_films_part1/sickle3-edofed')
f = sorted((ROOT/'280120-07').glob('*.tiff'))[5]

sam = sam_model_registry['vit_b'](checkpoint='/kaggle/working/sam_vit_b.pth')
sam.to('cuda')
gen = SamAutomaticMaskGenerator(sam, points_per_side=32, pred_iou_thresh=0.86,
                                stability_score_thresh=0.92,
                                min_mask_region_area=4000)

t0 = time.time()
rgb = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
masks = gen.generate(rgb)
areas = np.array([m['area'] for m in masks])
keep = areas[(areas >= 4000) & (areas <= 30000)]
print(f'{f.parent.name}/{f.stem}   {time.time()-t0:.0f}s')
print(f'  raw masks        {len(masks)}')
print(f'  kept 4k-30k px   {len(keep)}')
print(f'  median kept area {np.median(keep):.0f} px    (expect ~10,200)')
print(f'  coverage         {100*keep.sum()/(rgb.shape[0]*rgb.shape[1]):.1f}%'
      f'    (thresholding gave 28.7%)')

In [ ]:
import numpy as np

H, W = rgb.shape[:2]
sel = [m for m in masks if 4000 <= m['area'] <= 30000]

# how much do the kept masks overlap each other?
acc = np.zeros((H, W), np.uint16)
for m in sel:
    acc[m['segmentation']] += 1
print(f'union coverage      {100*(acc>0).mean():5.1f}%   (thresholding 28.7%)')
print(f'sum of areas        {100*acc.sum()/(H*W):5.1f}%')
print(f'pixels claimed >1x  {100*(acc>1).mean():5.1f}%   <- nested/duplicate masks')

# are the masks sitting on dark cells or on pale background?
g = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
_, bwo = cv2.threshold(cv2.GaussianBlur(g,(5,5),0), 127, 255,
                       cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)
fg = bwo > 0
inside = [(m['segmentation'] & fg).sum()/m['area'] for m in sel]
inside = np.array(inside)
print(f'\nmask overlap with Otsu foreground:')
print(f'   median  {np.median(inside):.2f}')
print(f'   <0.5    {(inside<0.5).sum():3d} masks  <- probably background')
print(f'   >0.9    {(inside>0.9).sum():3d} masks  <- solidly on cells')

# intensity check: cells are darker than background
mu_bg = g[~fg].mean()
dark = np.array([g[m['segmentation']].mean() for m in sel])
print(f'\nbackground grey {mu_bg:.0f}')
print(f'mask grey: median {np.median(dark):.0f}, '
      f'{(dark > mu_bg-10).sum()} masks as pale as background')

In [ ]:
def clean_masks(masks, fg, lo=4000, hi=30000, iou_thr=0.35, fg_thr=0.5):
    """Keep plausible single cells; suppress nested duplicates."""
    sel = [m for m in masks if lo <= m['area'] <= hi
           and (m['segmentation'] & fg).sum() / m['area'] >= fg_thr]
    sel.sort(key=lambda m: -m['predicted_iou'])     # best proposals first
    kept = []
    for m in sel:
        s = m['segmentation']
        dup = False
        for k in kept:
            inter = (s & k['segmentation']).sum()
            if inter == 0:
                continue
            # containment, not IoU: a nested rim/centre is small but inside
            if inter / min(m['area'], k['area']) > iou_thr:
                dup = True; break
        if not dup:
            kept.append(m)
    return kept

kept = clean_masks(masks, fg)
acc = np.zeros((H, W), np.uint16)
for m in kept: acc[m['segmentation']] += 1
ar = np.array([m['area'] for m in kept])
print(f'before NMS {len(sel)}  ->  after {len(kept)}')
print(f'union coverage     {100*(acc>0).mean():.1f}%   (thresholding 28.7%)')
print(f'pixels claimed >1x {100*(acc>1).mean():.1f}%   (was 12.1%)')
print(f'median area        {np.median(ar):.0f} px   (expect ~10,200)')
print(f'cells              {len(kept)}   (blob estimate was ~149)')

In [ ]:
import matplotlib.pyplot as plt

ar = np.array([m['area'] for m in kept])
order = np.argsort(ar)
picks = {'smallest': order[:4], 'median': order[len(order)//2-2:len(order)//2+2],
         'largest': order[-4:]}

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for r, (name, idx) in enumerate(picks.items()):
    for c, i in enumerate(idx):
        m = kept[i]; s = m['segmentation']
        ys, xs = np.where(s); pad = 25
        y0, y1 = max(0, ys.min()-pad), min(H, ys.max()+pad)
        x0, x1 = max(0, xs.min()-pad), min(W, xs.max()+pad)
        ax = axes[r, c]
        ax.imshow(rgb[y0:y1, x0:x1])
        ax.contour(s[y0:y1, x0:x1], levels=[.5], colors='lime', linewidths=1.4)
        ax.set_title(f'{name}  {m["area"]:.0f} px', fontsize=8)
        ax.axis('off')
plt.tight_layout(); plt.show()

print('area distribution')
for q in (5, 25, 50, 75, 95):
    print(f'  p{q:<3d} {np.percentile(ar, q):7.0f} px')
print(f'\nbelow 7,000 px: {(ar<7000).sum()} masks '
      f'({100*(ar<7000).mean():.0f}%)  <- fragments or genuinely small cells?')

In [ ]:
# do the smallest masks look like cells too?
ar = np.array([m['area'] for m in kept])
small = np.argsort(ar)[:8]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axes.ravel(), small):
    m = kept[i]; s = m['segmentation']
    ys, xs = np.where(s); pad = 25
    y0,y1 = max(0,ys.min()-pad), min(H,ys.max()+pad)
    x0,x1 = max(0,xs.min()-pad), min(W,xs.max()+pad)
    ax.imshow(rgb[y0:y1,x0:x1])
    ax.contour(s[y0:y1,x0:x1], levels=[.5], colors='lime', linewidths=1.4)
    ax.set_title(f'{m["area"]:.0f} px', fontsize=8); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np, cv2
import matplotlib.pyplot as plt
print('kept exists:', 'kept' in dir())
print('rgb  exists:', 'rgb'  in dir())

In [ ]:
from pathlib import Path
print('weights:', Path('/kaggle/working/sam_vit_b.pth').exists())
from segment_anything import sam_model_registry
print('package: OK')

In [ ]:
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth \
      -O /kaggle/working/sam_vit_b.pth

from pathlib import Path
p = Path('/kaggle/working/sam_vit_b.pth')
print(f'{p.stat().st_size/1e6:.0f} MB' if p.exists() else 'FAILED')

In [ ]:
# ==========================================================
# PAPER 2 - STAGE 2: GROUND-TRUTH INSTANCE MASKS
#
# SAM automatic mask generator with the parameters reported in
# Methods, plus two cleaning steps established by the sanity check:
#
#   1. reject masks whose overlap with Otsu foreground < 0.5
#      (SAM occasionally proposes background regions)
#   2. containment-based non-maximum suppression
#      (SAM emits nested proposals - a cell, its rim, its pallor
#       centre - which double-counted 12.1% of pixels and would
#       have inflated the recall denominator)
#
# Writes one int32 label array per field to /kaggle/working/gt_masks
# ==========================================================

import numpy as np, cv2, torch, time, json
from pathlib import Path
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/'
            'thin_films_part1/sickle3-edofed')
OUT  = Path('/kaggle/working/gt_masks'); OUT.mkdir(parents=True, exist_ok=True)
CKPT = '/kaggle/working/sam_vit_b.pth'

# SAM generator parameters (as reported)
PPS, IOU_T, STAB, MINREG = 32, 0.86, 0.92, 4000
# cleaning parameters
AREA_LO, AREA_HI = 4000, 30000      # plausible single erythrocyte
FG_FRAC          = 0.50             # min overlap with Otsu foreground
CONTAINMENT      = 0.35             # NMS: reject if this fraction of the
                                    # smaller mask lies inside a kept mask

print('loading SAM...', flush=True)
sam = sam_model_registry['vit_b'](checkpoint=CKPT)
sam.to('cuda' if torch.cuda.is_available() else 'cpu')
gen = SamAutomaticMaskGenerator(sam, points_per_side=PPS,
                                pred_iou_thresh=IOU_T,
                                stability_score_thresh=STAB,
                                min_mask_region_area=MINREG)

def otsu_fg(rgb):
    g = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    _, bw = cv2.threshold(g, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return bw > 0

def clean(masks, fg):
    """Area window, foreground check, then containment NMS."""
    sel = [m for m in masks
           if AREA_LO <= m['area'] <= AREA_HI
           and (m['segmentation'] & fg).sum() / m['area'] >= FG_FRAC]
    sel.sort(key=lambda m: -m['predicted_iou'])       # best proposals first
    kept = []
    for m in sel:
        s = m['segmentation']
        dup = False
        for k in kept:
            inter = (s & k['segmentation']).sum()
            if inter and inter / min(m['area'], k['area']) > CONTAINMENT:
                dup = True
                break
        if not dup:
            kept.append(m)
    return kept

# one field per slide, middle by filename sort
fields = []
for d in sorted(ROOT.iterdir()):
    if d.is_dir():
        t = sorted(d.glob('*.tiff'))
        if t:
            fields.append(t[len(t) // 2])
print(f'{len(fields)} fields, one per slide\n', flush=True)

summary = []
t_all = time.time()
for i, f in enumerate(fields, 1):
    t0 = time.time()
    rgb = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    fg = otsu_fg(rgb)
    raw = gen.generate(rgb)
    kept = clean(raw, fg)

    lab = np.zeros(rgb.shape[:2], np.int32)
    for k, m in enumerate(kept, 1):
        lab[m['segmentation']] = k
    np.save(OUT / f'{f.parent.name}.npy', lab)

    ar = np.array([m['area'] for m in kept]) if kept else np.array([0])
    rec = dict(slide=f.parent.name, field=f.stem, raw=len(raw), cells=len(kept),
               median_area=float(np.median(ar)),
               coverage=float(100 * (lab > 0).mean()),
               otsu_coverage=float(100 * fg.mean()), secs=time.time() - t0)
    summary.append(rec)
    print(f'  {i:2d}/{len(fields)} {rec["slide"]}: '
          f'{rec["raw"]:3d} raw -> {rec["cells"]:3d} cells, '
          f'median {rec["median_area"]:5.0f} px, '
          f'cov {rec["coverage"]:4.1f}% (otsu {rec["otsu_coverage"]:4.1f}%)  '
          f'{rec["secs"]:.0f}s', flush=True)

import pandas as pd
df = pd.DataFrame(summary)
df.to_csv('/kaggle/working/gt_summary.csv', index=False)

print(f'\ndone in {(time.time()-t_all)/60:.1f} min')
print(f'total ground-truth cells : {df.cells.sum():,}')
print(f'cells per field          : {df.cells.mean():.0f} '
      f'(range {df.cells.min()}-{df.cells.max()})')
print(f'median cell area         : {df.median_area.median():.0f} px')
print(f'SAM coverage             : {df.coverage.mean():.1f}%')
print(f'Otsu coverage            : {df.otsu_coverage.mean():.1f}%')
print('\nSAM coverage should sit slightly above Otsu: it finds faint cells')
print('thresholding misses. A field far outside these ranges is worth')
print('inspecting before it enters the audit.')

# save the exact configuration for the Methods section
json.dump(dict(points_per_side=PPS, pred_iou_thresh=IOU_T,
               stability_score_thresh=STAB, min_mask_region_area=MINREG,
               area_window=[AREA_LO, AREA_HI], fg_fraction=FG_FRAC,
               containment_nms=CONTAINMENT, n_fields=len(fields),
               total_cells=int(df.cells.sum())),
          open('/kaggle/working/gt_config.json', 'w'), indent=2)
print('\nwritten: gt_masks/, gt_summary.csv, gt_config.json')

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/gt_masks', 'zip', '/kaggle/working/gt_masks')
print('zipped')

In [ ]:
from pathlib import Path
w = Path('/kaggle/working')
print('exists:', w.exists())
for p in sorted(w.rglob('*'))[:30]:
    print('  ', p.relative_to(w), f'{p.stat().st_size/1e6:.1f} MB' if p.is_file() else '/')

In [ ]:
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth \
      -O /kaggle/working/sam_vit_b.pth
from pathlib import Path
print(f"{Path('/kaggle/working/sam_vit_b.pth').stat().st_size/1e6:.0f} MB")

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/gt_masks', 'zip', '/kaggle/working/gt_masks')
print('zipped — download from the Output panel NOW')

In [ ]:
from pathlib import Path
print('weights :', Path('/kaggle/working/sam_vit_b.pth').exists())
try:
    from segment_anything import sam_model_registry
    print('package : OK')
except Exception as e:
    print('package : MISSING —', e)
print('masks   :', Path('/kaggle/working/gt_masks').exists())

In [ ]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
print('package OK')


In [ ]:
import shutil
shutil.make_archive('/kaggle/working/gt_masks', 'zip', '/kaggle/working/gt_masks')
print('zipped')

In [ ]:
import numpy as np, cv2, torch, time, json, shutil
import pandas as pd
from pathlib import Path
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/'
            'thin_films_part1/sickle3-edofed')
OUT  = Path('/kaggle/working/gt_masks'); OUT.mkdir(parents=True, exist_ok=True)
CKPT = '/kaggle/working/sam_vit_b.pth'

PPS, IOU_T, STAB, MINREG = 32, 0.86, 0.92, 4000
AREA_LO, AREA_HI = 4000, 30000
FG_FRAC, CONTAINMENT = 0.50, 0.35

print('loading SAM...', flush=True)
sam = sam_model_registry['vit_b'](checkpoint=CKPT)
sam.to('cuda' if torch.cuda.is_available() else 'cpu')
gen = SamAutomaticMaskGenerator(sam, points_per_side=PPS,
                                pred_iou_thresh=IOU_T,
                                stability_score_thresh=STAB,
                                min_mask_region_area=MINREG)

def otsu_fg(rgb):
    g = cv2.GaussianBlur(cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY), (5, 5), 0)
    _, bw = cv2.threshold(g, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return bw > 0

def clean(masks, fg):
    sel = [m for m in masks
           if AREA_LO <= m['area'] <= AREA_HI
           and (m['segmentation'] & fg).sum() / m['area'] >= FG_FRAC]
    sel.sort(key=lambda m: -m['predicted_iou'])
    kept = []
    for m in sel:
        s = m['segmentation']
        dup = False
        for k in kept:
            inter = (s & k['segmentation']).sum()
            if inter and inter / min(m['area'], k['area']) > CONTAINMENT:
                dup = True
                break
        if not dup:
            kept.append(m)
    return kept

fields = []
for d in sorted(ROOT.iterdir()):
    if d.is_dir():
        t = sorted(d.glob('*.tiff'))
        if t:
            fields.append(t[len(t) // 2])
print(f'{len(fields)} fields, one per slide\n', flush=True)

summary = []
t_all = time.time()
for i, f in enumerate(fields, 1):
    t0 = time.time()
    rgb = cv2.cvtColor(cv2.imread(str(f)), cv2.COLOR_BGR2RGB)
    fg = otsu_fg(rgb)
    raw = gen.generate(rgb)
    kept = clean(raw, fg)
    lab = np.zeros(rgb.shape[:2], np.int32)
    for k, m in enumerate(kept, 1):
        lab[m['segmentation']] = k
    np.save(OUT / f'{f.parent.name}.npy', lab)
    ar = np.array([m['area'] for m in kept]) if kept else np.array([0])
    rec = dict(slide=f.parent.name, field=f.stem, raw=len(raw),
               cells=len(kept), median_area=float(np.median(ar)),
               coverage=float(100 * (lab > 0).mean()),
               otsu_coverage=float(100 * fg.mean()), secs=time.time() - t0)
    summary.append(rec)
    print(f'  {i:2d}/{len(fields)} {rec["slide"]}: '
          f'{rec["raw"]:3d} raw -> {rec["cells"]:3d} cells, '
          f'median {rec["median_area"]:5.0f} px, '
          f'cov {rec["coverage"]:4.1f}% (otsu {rec["otsu_coverage"]:4.1f}%)  '
          f'{rec["secs"]:.0f}s', flush=True)

df = pd.DataFrame(summary)
df.to_csv('/kaggle/working/gt_summary.csv', index=False)
json.dump(dict(points_per_side=PPS, pred_iou_thresh=IOU_T,
               stability_score_thresh=STAB, min_mask_region_area=MINREG,
               area_window=[AREA_LO, AREA_HI], fg_fraction=FG_FRAC,
               containment_nms=CONTAINMENT, n_fields=len(fields),
               total_cells=int(df.cells.sum())),
          open('/kaggle/working/gt_config.json', 'w'), indent=2)

print(f'\ndone in {(time.time()-t_all)/60:.1f} min')
print(f'total ground-truth cells : {df.cells.sum():,}')
print(f'cells per field          : {df.cells.mean():.0f} '
      f'(range {df.cells.min()}-{df.cells.max()})')
print(f'median cell area         : {df.median_area.median():.0f} px')
print(f'SAM coverage             : {df.coverage.mean():.1f}%')
print(f'Otsu coverage            : {df.otsu_coverage.mean():.1f}%')

shutil.make_archive('/kaggle/working/gt_masks', 'zip', '/kaggle/working/gt_masks')
print('\nzipped -> gt_masks.zip  (download from the Output panel now)')

In [ ]:
!pip install -q cellpose

In [ ]:
import numpy as np, cv2
from pathlib import Path
print('numpy', np.__version__, '| cv2', cv2.__version__)

try:
    from cellpose import models
    print('cellpose OK')
except Exception as e:
    print('cellpose FAILED —', str(e)[:120])

m = sorted(Path('/kaggle/working/gt_masks').glob('*.npy'))
print(f'masks: {len(m)}')
if m:
    a = np.load(m[0])
    print(f'  {m[0].stem}: {len([i for i in np.unique(a) if i>0])} cells, {a.shape}')

In [ ]:
# ==========================================================
# PAPER 2 - STAGE 3: THE AUDIT
#
# Four segmenters scored against identical ground truth:
#   original   faithful reimplementation of rbc_segmentation()
#              from the released segment_rbcs.py
#   fix_A      keep the watershed instance labels; apply the same
#              [5000, 17000] px filter per REGION instead of per
#              connected component of their union
#   fix_B      fix_A + per-component seed thresholding (the released
#              code uses 0.3 x GLOBAL distance-transform max, so one
#              large clump raises the bar for the whole field)
#   cellpose   Cellpose-SAM zero-shot, if installed
#
# Beyond touching-vs-isolated, this measures CONTACT FRACTION per
# ground-truth cell (the share of its boundary abutting a neighbour)
# so recall can be reported as a dose-response against overlap depth.
# ==========================================================

import numpy as np, cv2, pandas as pd, time, json
from pathlib import Path
from scipy.ndimage import binary_fill_holes

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/'
            'thin_films_part1/sickle3-edofed')
GT   = Path('/kaggle/working/gt_masks')
OUT  = Path('/kaggle/working')

MIN_SIZE, MAX_SIZE = 5000, 17000     # the released thresholds
IOU_TAU = 0.5
USE_CELLPOSE = True                  # set False to skip

# ---------------- released pipeline, stage by stage ----------------
def _front(img):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (5, 5), 0)
    _, th = cv2.threshold(g, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = np.ones((3, 3), np.uint8)
    opening = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=2)
    sure_bg = cv2.dilate(opening, k, iterations=2)
    dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    return opening, sure_bg, dist

def _seed_global(dist, frac=0.3):
    _, fg = cv2.threshold(dist, frac * dist.max(), 255, 0)
    return np.uint8(fg)

def _seed_local(dist, opening, frac=0.3):
    n, lab = cv2.connectedComponents(opening.astype(np.uint8), 8)
    fg = np.zeros(dist.shape, np.uint8)
    for i in range(1, n):
        m = lab == i
        mx = dist[m].max()
        if mx > 0:
            fg[m & (dist > frac * mx)] = 255
    return fg

def _ws(img, sure_bg, sure_fg):
    unknown = cv2.subtract(sure_bg, sure_fg)
    _, mk = cv2.connectedComponents(sure_fg)
    mk = mk + 1
    mk[unknown == 255] = 0
    return cv2.watershed(img, mk)

def seg_original(img, tr=None):
    opening, sure_bg, dist = _front(img)
    mk = _ws(img, sure_bg, _seed_global(dist))
    n_ws = len([l for l in np.unique(mk) if l > 1])
    bw = binary_fill_holes(mk > 1).astype(np.uint8)      # labels discarded
    nlab, lab, st, _ = cv2.connectedComponentsWithStats(bw, 4)
    keep = np.zeros(bw.shape, np.uint8)
    n_del_big = n_del_small = 0
    for i in range(1, nlab):
        a = st[i, cv2.CC_STAT_AREA]
        if a < MIN_SIZE:   n_del_small += 1; continue
        if a >= MAX_SIZE:  n_del_big += 1;   continue
        keep[lab == i] = 1
    n_out, lab_out = cv2.connectedComponents(keep, 8)
    if tr is not None:
        tr.update(ws_regions=n_ws, components=nlab - 1, kept=n_out - 1,
                  deleted_oversize=n_del_big, deleted_undersize=n_del_small)
    return lab_out.astype(np.int32)

def _seg_fix(img, tr=None, local=False):
    opening, sure_bg, dist = _front(img)
    sf = _seed_local(dist, opening) if local else _seed_global(dist)
    mk = _ws(img, sure_bg, sf)
    out = np.zeros(mk.shape, np.int32)
    nxt = small = big = 0
    for lb in np.unique(mk):
        if lb <= 1: continue
        reg = binary_fill_holes(mk == lb)
        a = int(reg.sum())
        if a < MIN_SIZE:  small += 1; continue
        if a >= MAX_SIZE: big += 1;   continue
        nxt += 1
        out[reg] = nxt
    if tr is not None:
        tr.update(ws_regions=len([l for l in np.unique(mk) if l > 1]),
                  kept=nxt, deleted_oversize=big, deleted_undersize=small)
    return out

def seg_fix_a(img, tr=None): return _seg_fix(img, tr, local=False)
def seg_fix_b(img, tr=None): return _seg_fix(img, tr, local=True)

CP = None
def seg_cellpose(img, tr=None):
    global CP
    if CP is None:
        from cellpose import models
        CP = models.CellposeModel(gpu=True)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:
        m, _, _ = CP.eval(rgb, diameter=115)
    except TypeError:
        m, _, _, _ = CP.eval(rgb, diameter=115, channels=[0, 0])
    m = np.asarray(m).astype(np.int32)
    if tr is not None:
        tr.update(kept=len([i for i in np.unique(m) if i > 0]))
    return m

# ---------------- contact fraction per ground-truth cell ----------------
def contact_fraction(lab):
    """Share of each instance's boundary that abuts another instance."""
    L = lab.astype(np.int32); H, W = L.shape
    pad = np.zeros((H + 2, W + 2), np.int32); pad[1:-1, 1:-1] = L
    bnd = np.zeros_like(L, bool); tch = np.zeros_like(L, bool)
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == dx == 0: continue
            nb = pad[1 + dy:H + 1 + dy, 1 + dx:W + 1 + dx]
            d = (nb != L) & (L > 0)
            bnd |= d
            tch |= d & (nb > 0)
    n = int(L.max()) + 1
    b = np.bincount(L[bnd], minlength=n)
    t = np.bincount(L[tch], minlength=n)
    return {int(i): float(t[i] / max(b[i], 1)) for i in np.unique(L[L > 0])}

def score(gt, pred, tau=IOU_TAU):
    pa = {int(i): int((pred == i).sum()) for i in np.unique(pred) if i > 0}
    out = {}
    for gi in [x for x in np.unique(gt) if x > 0]:
        m = gt == gi; ga = int(m.sum())
        best = 0.0
        vals, cnts = np.unique(pred[m], return_counts=True)
        for pi, inter in zip(vals, cnts):
            if pi == 0: continue
            best = max(best, inter / (ga + pa[int(pi)] - inter))
        out[int(gi)] = best
    return out

# ---------------- run ----------------
METHODS = {'original': seg_original, 'fix_A_labels': seg_fix_a,
           'fix_B_labels_local_seeds': seg_fix_b}
if USE_CELLPOSE:
    METHODS['cellpose_sam'] = seg_cellpose

gt_files = sorted(GT.glob('*.npy'))
print(f'{len(gt_files)} fields with ground truth\n', flush=True)

rows, traces = [], []
for i, gf in enumerate(gt_files, 1):
    slide = gf.stem
    tiffs = sorted((ROOT / slide).glob('*.tiff'))
    field = tiffs[len(tiffs) // 2]
    img = cv2.imread(str(field))
    gt = np.load(gf)
    cf = contact_fraction(gt)
    areas = {int(x): int((gt == x).sum()) for x in np.unique(gt) if x > 0}

    for name, fn in METHODS.items():
        tr = dict(slide=slide, method=name)
        t0 = time.time()
        pred = fn(img, tr)
        tr['secs'] = time.time() - t0
        traces.append(tr)
        for gi, iou in score(gt, pred).items():
            rows.append(dict(slide=slide, method=name, inst=gi, iou=iou,
                             recovered=int(iou >= IOU_TAU),
                             contact=cf.get(gi, 0.0), area=areas[gi]))
    print(f'  {i:2d}/{len(gt_files)} {slide}: {len(cf)} gt cells', flush=True)

df = pd.DataFrame(rows); df.to_csv(OUT / 'audit_cells.csv', index=False)
tf = pd.DataFrame(traces); tf.to_csv(OUT / 'audit_trace.csv', index=False)

# ---------------- report ----------------
df['touching'] = df.contact > 0.01
print('\n' + '=' * 72)
print('RECALL AT IoU 0.5')
print('=' * 72)
for name in METHODS:
    g = df[df.method == name]
    iso, tou = g[~g.touching], g[g.touching]
    print(f'{name:26s} overall {100*g.recovered.mean():5.1f}   '
          f'isolated {100*iso.recovered.mean():5.1f}   '
          f'touching {100*tou.recovered.mean():5.1f}   n={len(g)}')

print('\n' + '=' * 72)
print('RECALL vs CONTACT FRACTION  (dose-response)')
print('=' * 72)
CB = [-.001, .01, .05, .10, .20, .35, .50, 1.0]
df['cbin'] = pd.cut(df.contact, CB)
piv = df.pivot_table(index='cbin', columns='method', values='recovered',
                     aggfunc='mean', observed=True)
cnt = df[df.method == 'original'].groupby('cbin', observed=True).size().rename('n')
print(pd.concat([cnt, piv], axis=1).to_string(float_format=lambda v: f'{v:.3f}'))

print('\n' + '=' * 72)
print('STAGE TRACE  (mean per field)')
print('=' * 72)
print(tf.groupby('method').mean(numeric_only=True).to_string(
    float_format=lambda v: f'{v:.1f}'))
print('\nws_regions = cells watershed resolved; kept = cells that survived.')
print('For "original" the gap is the cost of discarding the instance labels.')

o = df[df.method == 'original']
for name in [m for m in METHODS if m != 'original']:
    n_ = df[df.method == name]
    print(f'\n{name}: {100*(n_.recovered.mean()-o.recovered.mean()):+.1f} pts overall, '
          f'{100*(n_[n_.touching].recovered.mean()-o[o.touching].recovered.mean()):+.1f} pts on touching cells')

# crowding correlation
from scipy.stats import spearmanr
per = df[df.method == 'original'].groupby('slide').agg(
    recall=('recovered', 'mean'), crowd=('touching', 'mean'))
r, p = spearmanr(per.crowd, per.recall)
print(f'\ncrowding vs recall: Spearman rho = {r:.2f}, p = {p:.4f}, n = {len(per)}')
print('(paper reported rho = -0.60, p = 0.023, n = 14)')

25 fields with ground truth



   1/25 280120-07: 192 gt cells
   2/25 280120-08: 111 gt cells
   3/25 280120-09: 130 gt cells
   4/25 280120-10: 154 gt cells
   5/25 280120-11: 168 gt cells
   6/25 280120-12: 163 gt cells
   7/25 280120-13: 167 gt cells
   8/25 280120-14: 162 gt cells
   9/25 280120-15: 125 gt cells
  10/25 280120-18: 132 gt cells
  11/25 280120-19: 145 gt cells
  12/25 280120-20: 174 gt cells
  13/25 280120-21: 188 gt cells


In [38]:
import os, time, numpy as np
from pathlib import Path

w = Path('/kaggle/working')
print('files in working:')
for p in sorted(w.glob('*.npy')) + sorted(w.glob('*.csv')):
    age = (time.time() - p.stat().st_mtime) / 60
    print(f'  {p.name:32s} {p.stat().st_size/1e6:7.1f} MB   {age:5.1f} min old')

for n in ['human', 'sam_gt', 'preds', 'img']:
    print(f'{n:8s} in memory:', n in dir())
if 'preds' in dir():
    print('  segmented:', list(preds.keys()))

try:
    import cellpose; print('\ncellpose:', cellpose.version)
except Exception as e:
    print('\ncellpose:', e)

files in working:
  human_080119-07.npy                 22.1 MB    42.8 min old
  sam_080119-07.npy                   22.1 MB    42.8 min old
  audit_cells.csv                      0.9 MB   101.8 min old
  audit_cells_full.csv                 1.0 MB   101.8 min old
  audit_per_field.csv                  0.0 MB   101.8 min old
  audit_trace.csv                      0.0 MB   101.8 min old
  audit_trace_full.csv                 0.0 MB   101.8 min old
  field_coverage.csv                   0.0 MB    69.2 min old
  gt_shape.csv                         0.3 MB    56.6 min old
  gt_summary.csv                       0.0 MB   101.8 min old
  slide_labels.csv                     0.0 MB    77.4 min old
  unreadable_tiffs.csv                 0.0 MB    71.8 min old
human    in memory: True
sam_gt   in memory: True
preds    in memory: True
img      in memory: True
  segmented: ['original', 'fix_A']

cellpose: 4.2.1.1


In [39]:
import numpy as np, pandas as pd, cv2

if 'cellpose_sam' not in preds:
    from cellpose import models
    cp = models.CellposeModel(gpu=True)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    try:    m, _, _ = cp.eval(rgb, diameter=115)
    except TypeError: m, _, _, _ = cp.eval(rgb, diameter=115, channels=[0, 0])
    preds['cellpose_sam'] = np.asarray(m).astype(np.int32)
    print('segmented: cellpose_sam')

for gtname, gt in [('HUMAN', human), ('SAM', sam_gt)]:
    ecc, cf = shape_stats(gt), contact(gt)
    A = pd.concat([score(gt, p, ecc, cf).assign(method=n)
                   for n, p in preds.items()])
    A['eq'] = pd.qcut(A.ecc, 5, labels=['round','q2','q3','q4','elongated'])
    print('\n' + '=' * 66)
    print(f'{gtname} GROUND TRUTH - recall by elongation quintile')
    print('=' * 66)
    for label, D in [('all cells', A), ('isolated only', A[A.contact <= 0.01])]:
        piv = D.pivot_table(index='eq', columns='method', values='rec',
                            aggfunc='mean', observed=True)
        n = D[D.method == 'original'].groupby('eq', observed=True).size().rename('n')
        print(f'\n  {label}:')
        print(pd.concat([n, piv], axis=1).to_string(float_format=lambda v: f'{v:.3f}'))

100%|██████████| 1.15G/1.15G [00:11<00:00, 108MB/s]    


segmented: cellpose_sam

HUMAN GROUND TRUTH - recall by elongation quintile

  all cells:
            n  cellpose_sam  fix_A  original
eq                                          
round      48         1.000  0.667     0.500
q2         48         1.000  0.604     0.500
q3         48         0.979  0.562     0.458
q4         47         0.936  0.489     0.362
elongated  48         0.854  0.312     0.250

  isolated only:
            n  cellpose_sam  fix_A  original
eq                                          
round      32         1.000  0.656     0.562
q2         35         1.000  0.686     0.600
q3         33         1.000  0.667     0.636
q4         28         1.000  0.679     0.500
elongated  32         0.938  0.375     0.375

SAM GROUND TRUTH - recall by elongation quintile

  all cells:
            n  cellpose_sam  fix_A  original
eq                                          
round      45         1.000  0.644     0.489
q2         45         1.000  0.600     0.489
q3         45     

In [40]:
import shutil, os
os.makedirs('/kaggle/working/human_verify', exist_ok=True)
for f in ['human_080119-07.npy', 'sam_080119-07.npy']:
    shutil.copy(f'/kaggle/working/{f}', f'/kaggle/working/human_verify/{f}')
shutil.make_archive('/kaggle/working/human_verify', 'zip', '/kaggle/working/human_verify')
print('zipped -> human_verify.zip')

zipped -> human_verify.zip


In [41]:
import numpy as np, cv2, pandas as pd, time
from pathlib import Path
from scipy.ndimage import binary_fill_holes

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/'
            'thin_films_part1/sickle3-edofed')
GT   = Path('/kaggle/working/gt_masks')
if not GT.exists():
    GT = next(p for p in Path('/kaggle/input').rglob('gt_masks') if p.is_dir())
print('gt from:', GT)

MIN_SIZE, MAX_SIZE = 5000, 17000
TAUS = [0.3, 0.5, 0.7, 0.75]

def _front(im):
    g = cv2.GaussianBlur(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, th = cv2.threshold(g, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = np.ones((3, 3), np.uint8)
    op = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=2)
    return op, cv2.dilate(op, k, iterations=2), cv2.distanceTransform(op, cv2.DIST_L2, 5)

def _ws(im, bg, sfg):
    unk = cv2.subtract(bg, sfg)
    _, mk = cv2.connectedComponents(sfg); mk += 1; mk[unk == 255] = 0
    return cv2.watershed(im, mk)

def _seed_global(dist, frac=.3):
    _, fg = cv2.threshold(dist, frac * dist.max(), 255, 0); return np.uint8(fg)

def _seed_local(dist, op, frac=.3):
    n, lab = cv2.connectedComponents(op.astype(np.uint8), 8)
    fg = np.zeros(dist.shape, np.uint8)
    for i in range(1, n):
        m = lab == i; mx = dist[m].max()
        if mx > 0: fg[m & (dist > frac * mx)] = 255
    return fg

def seg_original(im):
    op, bg, dist = _front(im)
    mk = _ws(im, bg, _seed_global(dist))
    bw = binary_fill_holes(mk > 1).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(bw, 4)
    keep = np.zeros(bw.shape, np.uint8)
    for i in range(1, n):
        if MIN_SIZE <= st[i, cv2.CC_STAT_AREA] < MAX_SIZE:
            keep[lab == i] = 1
    return cv2.connectedComponents(keep, 8)[1].astype(np.int32)

def _fix(im, local):
    op, bg, dist = _front(im)
    mk = _ws(im, bg, _seed_local(dist, op) if local else _seed_global(dist))
    out = np.zeros(mk.shape, np.int32); nx = 0
    for lb in np.unique(mk):
        if lb <= 1: continue
        r = binary_fill_holes(mk == lb); a = int(r.sum())
        if MIN_SIZE <= a < MAX_SIZE:
            nx += 1; out[r] = nx
    return out

CP = [None]
def seg_cellpose(im):
    if CP[0] is None:
        from cellpose import models
        CP[0] = models.CellposeModel(gpu=True)
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    try:    m, _, _ = CP[0].eval(rgb, diameter=115)
    except TypeError: m, _, _, _ = CP[0].eval(rgb, diameter=115, channels=[0, 0])
    return np.asarray(m).astype(np.int32)

METHODS = {'original': seg_original,
           'fix_A':    lambda im: _fix(im, False),
           'fix_B':    lambda im: _fix(im, True),
           'cellpose_sam': seg_cellpose}

def iou_matrix(gt, pred):
    """best-IoU per gt cell and per predicted cell, one pass."""
    ga = {int(i): int((gt == i).sum()) for i in np.unique(gt) if i > 0}
    pa = {int(i): int((pred == i).sum()) for i in np.unique(pred) if i > 0}
    best_g = {i: 0.0 for i in ga}
    best_p = {j: 0.0 for j in pa}
    both = (gt > 0) & (pred > 0)
    if both.any():
        pairs, cnt = np.unique(np.stack([gt[both], pred[both]]), axis=1,
                               return_counts=True)
        for (gi, pj), inter in zip(pairs.T, cnt):
            gi, pj = int(gi), int(pj)
            iou = inter / (ga[gi] + pa[pj] - inter)
            if iou > best_g[gi]: best_g[gi] = iou
            if iou > best_p[pj]: best_p[pj] = iou
    return best_g, best_p, len(ga), len(pa)

rows = []
gts = sorted(GT.glob('*.npy'))
for i, gf in enumerate(gts, 1):
    slide = gf.stem
    tif = sorted((ROOT / slide).glob('*.tiff'))
    im = cv2.imread(str(tif[len(tif)//2]))
    gt = np.load(gf)
    for name, fn in METHODS.items():
        t0 = time.time()
        pred = fn(im)
        bg, bp, n_gt, n_pd = iou_matrix(gt, pred)
        rec = dict(slide=slide, method=name, n_gt=n_gt, n_pred=n_pd,
                   secs=time.time() - t0)
        for t in TAUS:
            tp_g = sum(v >= t for v in bg.values())
            tp_p = sum(v >= t for v in bp.values())
            rec[f'recall@{t}']    = tp_g / max(n_gt, 1)
            rec[f'precision@{t}'] = tp_p / max(n_pd, 1)
            rec[f'f1@{t}'] = (2 * tp_g * tp_p / max(tp_g + tp_p, 1)
                              / max(n_gt, 1) if tp_g + tp_p else 0)
        rows.append(rec)
    print(f'  {i:2d}/{len(gts)} {slide}', flush=True)

pr = pd.DataFrame(rows)
pr.to_csv('/kaggle/working/audit_precision.csv', index=False)

print('\n' + '=' * 72)
print('PRECISION / RECALL / F1  (micro-averaged over 25 fields)')
print('=' * 72)
for t in TAUS:
    print(f'\n  IoU {t}')
    print(f'  {"method":14s} {"recall":>8s} {"prec":>8s} {"F1":>8s} '
          f'{"pred/field":>11s}')
    for m in METHODS:
        g = pr[pr.method == m]
        R = (g[f'recall@{t}'] * g.n_gt).sum() / g.n_gt.sum()
        P = (g[f'precision@{t}'] * g.n_pred).sum() / g.n_pred.sum()
        F = 2 * P * R / (P + R) if P + R else 0
        print(f'  {m:14s} {100*R:7.1f}% {100*P:7.1f}% {100*F:7.1f}% '
              f'{g.n_pred.mean():10.1f}')

print('\n' + '=' * 72)
print('FALSE POSITIVE RATE AT IoU 0.5')
print('=' * 72)
for m in METHODS:
    g = pr[pr.method == m]
    fp = (1 - g['precision@0.5']) * g.n_pred
    print(f'  {m:14s} {fp.sum():6.0f} spurious of {g.n_pred.sum():6.0f} '
          f'predictions  ({100*fp.sum()/g.n_pred.sum():.1f}%)')
print('\nwritten: audit_precision.csv')

gt from: /kaggle/working/gt_masks
   1/25 280120-07
   2/25 280120-08
   3/25 280120-09
   4/25 280120-10
   5/25 280120-11
   6/25 280120-12
   7/25 280120-13
   8/25 280120-14
   9/25 280120-15
  10/25 280120-18
  11/25 280120-19
  12/25 280120-20
  13/25 280120-21
  14/25 280120-22
  15/25 280120-24
  16/25 280120-26
  17/25 280120-29
  18/25 280120-30
  19/25 280120-31
  20/25 280120-35
  21/25 280120-36
  22/25 280120-38
  23/25 280120-39
  24/25 280120-40
  25/25 280120-41

PRECISION / RECALL / F1  (micro-averaged over 25 fields)

  IoU 0.3
  method           recall     prec       F1  pred/field
  original          58.5%    99.9%    73.8%       85.6
  fix_A             73.4%    99.3%    84.4%      110.0
  fix_B             74.1%    99.9%    85.0%      108.8
  cellpose_sam      97.6%    97.7%    97.7%      152.1

  IoU 0.5
  method           recall     prec       F1  pred/field
  original          55.7%    97.9%    71.0%       85.6
  fix_A             68.9%    94.3%    79.6%      

In [ ]:
import numpy as np, pandas as pd

shp = pd.read_csv('/kaggle/working/gt_shape.csv')
df  = pd.read_csv('/kaggle/working/audit_cells_full.csv')
m = df.merge(shp, on=['slide', 'inst'], suffixes=('', '_s'))

print('=' * 74)
print('DOWNSTREAM ERROR: SICKLE FRACTION, TRUE vs PIPELINE-REPORTED')
print('=' * 74)
print('Sickled defined by eccentricity threshold; swept because no')
print('single threshold is canonical.\n')

for thr in [0.60, 0.65, 0.70, 0.75, 0.80]:
    gt_cells = m[m.method == 'original'][['slide', 'inst', 'ecc']].drop_duplicates()
    true_frac = (gt_cells.ecc >= thr).mean()
    print(f'  eccentricity >= {thr:.2f}   true sickle fraction {100*true_frac:5.1f}%')
    for meth in ['original', 'fix_A', 'fix_B', 'cellpose_sam']:
        g = m[m.method == meth]
        kept = g[g['rec@0.5'] == 1]
        obs = (kept.ecc >= thr).mean()
        print(f'      {meth:14s} reported {100*obs:5.1f}%   '
              f'error {100*(obs-true_frac):+5.1f} pp   '
              f'relative {100*(obs-true_frac)/true_frac:+6.1f}%')
    print()

print('=' * 74)
print('PER-SLIDE ERROR (eccentricity >= 0.70), original')
print('=' * 74)
THR = 0.70
rows = []
for sl, g in m[m.method == 'original'].groupby('slide'):
    true = (g.ecc >= THR).mean()
    obs  = (g[g['rec@0.5'] == 1].ecc >= THR).mean()
    rows.append(dict(slide=sl, true=100*true, reported=100*obs,
                     error=100*(obs - true)))
pf = pd.DataFrame(rows)
print(pf.describe().loc[['mean', 'std', 'min', 'max']].to_string(
      float_format=lambda v: f'{v:.1f}'))
print(f'\nslides where the pipeline under-reports: '
      f'{(pf.error < 0).sum()} of {len(pf)}')

DOWNSTREAM ERROR: SICKLE FRACTION, TRUE vs PIPELINE-REPORTED
Sickled defined by eccentricity threshold; swept because no
single threshold is canonical.

  eccentricity >= 0.60   true sickle fraction  48.4%
      original       reported  42.0%   error  -6.5 pp   relative  -13.3%
      fix_A          reported  45.9%   error  -2.5 pp   relative   -5.2%
      fix_B          reported  42.9%   error  -5.5 pp   relative  -11.3%
      cellpose_sam   reported  47.2%   error  -1.2 pp   relative   -2.5%

  eccentricity >= 0.65   true sickle fraction  38.9%
      original       reported  33.0%   error  -5.9 pp   relative  -15.2%
      fix_A          reported  36.5%   error  -2.4 pp   relative   -6.2%
      fix_B          reported  33.8%   error  -5.1 pp   relative  -13.2%
      cellpose_sam   reported  37.6%   error  -1.3 pp   relative   -3.5%

  eccentricity >= 0.70   true sickle fraction  30.9%
      original       reported  25.6%   error  -5.3 pp   relative  -17.1%
      fix_A          reported

In [ ]:
import numpy as np, cv2, pandas as pd, time
from pathlib import Path
from scipy.ndimage import binary_fill_holes
from skimage.morphology import h_maxima
from skimage.measure import label as sklabel
from skimage.segmentation import watershed as skws

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/'
            'thin_films_part1/sickle3-edofed')
GT = Path('/kaggle/working/gt_masks')
MIN_SIZE, MAX_SIZE, TAU = 5000, 17000, 0.5
H_GRID = [3, 5, 7, 9, 12]

def _front(im):
    g = cv2.GaussianBlur(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, th = cv2.threshold(g, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = np.ones((3, 3), np.uint8)
    op = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=2)
    return op, cv2.dilate(op, k, iterations=2), cv2.distanceTransform(op, cv2.DIST_L2, 5)

def seg_fix_c(im, h):
    op, bg, dist = _front(im)
    fg = op > 0
    seeds = sklabel(h_maxima(dist, h) > 0)
    if seeds.max() == 0:
        return np.zeros(fg.shape, np.int32)
    lab = skws(-dist, seeds, mask=fg)
    out = np.zeros(lab.shape, np.int32); nx = 0
    for i in range(1, lab.max() + 1):
        r = binary_fill_holes(lab == i); a = int(r.sum())
        if MIN_SIZE <= a < MAX_SIZE:
            nx += 1; out[r] = nx
    return out

def score_pair(gt, pred):
    ga = {int(i): int((gt == i).sum()) for i in np.unique(gt) if i > 0}
    pa = {int(i): int((pred == i).sum()) for i in np.unique(pred) if i > 0}
    bg = {i: 0.0 for i in ga}; bp = {j: 0.0 for j in pa}
    both = (gt > 0) & (pred > 0)
    if both.any():
        pr_, cnt = np.unique(np.stack([gt[both], pred[both]]), axis=1,
                             return_counts=True)
        for (gi, pj), it in zip(pr_.T, cnt):
            gi, pj = int(gi), int(pj)
            v = it / (ga[gi] + pa[pj] - it)
            bg[gi] = max(bg[gi], v); bp[pj] = max(bp[pj], v)
    return bg, bp, len(ga), len(pa)

gts = sorted(GT.glob('*.npy'))
print(f'{len(gts)} gt fields\n')

sweep = []
for h in H_GRID:
    tp_g = tp_p = n_g = n_p = 0
    t0 = time.time()
    for gf in gts:
        tif = sorted((ROOT / gf.stem).glob('*.tiff'))
        im = cv2.imread(str(tif[len(tif)//2])); gt = np.load(gf)
        bg, bp, ng, npd = score_pair(gt, seg_fix_c(im, h))
        tp_g += sum(v >= TAU for v in bg.values()); n_g += ng
        tp_p += sum(v >= TAU for v in bp.values()); n_p += npd
    R, P = tp_g/n_g, tp_p/max(n_p, 1)
    sweep.append(dict(h=h, recall=R, precision=P,
                      f1=2*P*R/(P+R) if P+R else 0,
                      pred_per_field=n_p/len(gts)))
    print(f'  h={h:2d}  recall {100*R:5.1f}%  prec {100*P:5.1f}%  '
          f'F1 {100*sweep[-1]["f1"]:5.1f}%  {n_p/len(gts):5.1f} pred/field  '
          f'({time.time()-t0:.0f}s)', flush=True)

sw = pd.DataFrame(sweep); sw.to_csv('/kaggle/working/fixC_sweep.csv', index=False)
BEST = int(sw.loc[sw.f1.idxmax(), 'h'])
print(f'\nbest h = {BEST} by F1')

shp = pd.read_csv('/kaggle/working/gt_shape.csv')
rows = []
for gf in gts:
    tif = sorted((ROOT / gf.stem).glob('*.tiff'))
    im = cv2.imread(str(tif[len(tif)//2])); gt = np.load(gf)
    bg, _, _, _ = score_pair(gt, seg_fix_c(im, BEST))
    L = gt.astype(np.int32); hh, ww = L.shape
    pad = np.zeros((hh+2, ww+2), np.int32); pad[1:-1, 1:-1] = L
    bnd = np.zeros_like(L, bool); tch = np.zeros_like(L, bool)
    for dy in (-1, 0, 1):
        for dx in (-1, 0, 1):
            if dy == dx == 0: continue
            nb = pad[1+dy:hh+1+dy, 1+dx:ww+1+dx]
            d = (nb != L) & (L > 0); bnd |= d; tch |= d & (nb > 0)
    n = int(L.max())+1
    b = np.bincount(L[bnd], minlength=n); t = np.bincount(L[tch], minlength=n)
    for i, v in bg.items():
        rows.append(dict(slide=gf.stem, inst=i, iou=v,
                         **{'rec@0.5': int(v >= TAU)},
                         contact=float(t[i]/max(b[i], 1))))
fc = pd.DataFrame(rows).merge(shp, on=['slide', 'inst'])
fc.to_csv('/kaggle/working/fixC_cells.csv', index=False)

print('\n' + '=' * 70)
print(f'fix_C (h={BEST}) - RECALL BY ELONGATION, ISOLATED CELLS ONLY')
print('=' * 70)
iso = fc[fc.contact <= 0.01].copy()
iso['eq'] = pd.qcut(iso.ecc, 5, labels=['round','q2','q3','q4','elongated'])
tab = iso.groupby('eq', observed=True).agg(n=('rec@0.5','size'),
                                           recall=('rec@0.5','mean'))
print(tab.to_string(float_format=lambda v: f'{v:.3f}'))
print(f'\nround->elongated drop: '
      f'{100*(tab.recall.iloc[0]-tab.recall.iloc[-1]):.1f} pp'
      '   (original: 20.4 pp)')

print('\n' + '=' * 70)
print('DOWNSTREAM SICKLE FRACTION ERROR')
print('=' * 70)
for thr in [0.65, 0.70, 0.75]:
    true = (fc.ecc >= thr).mean()
    obs = (fc[fc['rec@0.5'] == 1].ecc >= thr).mean()
    print(f'  ecc >= {thr:.2f}   true {100*true:5.1f}%   '
          f'fix_C {100*obs:5.1f}%   error {100*(obs-true):+5.1f} pp')
print('\n  (original -5.3, fix_B -4.9, cellpose -1.3 at ecc 0.70)')

In [ ]:
import shutil, os
from pathlib import Path

w = Path('/kaggle/working')
out = w / 'paper2_all'; out.mkdir(exist_ok=True)

for name in ['audit_cells_full.csv', 'audit_trace_full.csv', 'audit_per_field.csv',
             'audit_cells.csv', 'audit_trace.csv', 'audit_precision.csv',
             'gt_shape.csv', 'gt_summary.csv', 'gt_config.json',
             'slide_labels.csv', 'field_coverage.csv', 'unreadable_tiffs.csv',
             'human_080119-07.npy', 'sam_080119-07.npy']:
    p = w / name
    print(('saved   ' if p.exists() else 'MISSING '), name)
    if p.exists():
        shutil.copy(p, out / name)

if (w / 'gt_masks').exists():
    shutil.make_archive(str(out / 'gt_masks'), 'zip', str(w / 'gt_masks'))
    print('saved    gt_masks.zip')

shutil.make_archive(str(w / 'paper2_all'), 'zip', str(out))
print('\n->', (w / 'paper2_all.zip').stat().st_size / 1e6, 'MB')

In [ ]:
from pathlib import Path
for p in sorted(Path('/kaggle/input').rglob('*')):
    if p.is_file() and 'thin-flims' not in str(p) and 'erythrocyte' not in str(p):
        print(f'{p.stat().st_size/1e6:8.1f} MB  {p.relative_to("/kaggle/input")}')

In [ ]:
from pathlib import Path
d = Path(r'C:\path\to\keep')     # your DEST folder
from collections import Counter
c = Counter(f.name[:6] for f in d.iterdir() if f.is_dir())
for k, v in sorted(c.items()):
    print(k, v)
print('total folders:', sum(c.values()))

In [1]:
import subprocess, torch

print('torch sees CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print(f'memory allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB')

try:
    out = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(out.stdout)
except FileNotFoundError:
    print('no nvidia-smi - running CPU-only')

torch sees CUDA: False
no nvidia-smi - running CPU-only


In [2]:
import torch, pandas as pd
from pathlib import Path
from collections import Counter

print('CUDA:', torch.cuda.is_available(), '(should be False now)')

for d in sorted(Path('/kaggle/input').iterdir()):
    print(' ', d.name)

c = Counter()
for d in Path('/kaggle/input').rglob('*'):
    if d.is_dir() and any(d.glob('*.tif*')):
        c[d.name[:6]] += 1
print('\nslides by date:')
for k, v in sorted(c.items()):
    print(f'  {k}  {v}')

CUDA: False (should be False now)
  datasets

slides by date:
  280120  25


In [1]:
"""
FIG 1 - STUDY DESIGN AND METHODS OVERVIEW
(a) cohort construction  (b) ground-truth generation
(c) the five segmenters, identical except at the seeding step
"""
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pathlib import Path

OUT = Path('/kaggle/working/figures'); OUT.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 600,
    'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 7.5, 'savefig.bbox': 'tight', 'savefig.pad_inches': .03,
    'pdf.fonttype': 42, 'ps.fonttype': 42})

MM = 1/25.4
POS, NEG = '#D55E00', '#0072B2'
GTC, SHARED = '#009E73', '#E8E8E8'
SEED, EVAL = '#E69F00', '#D4D4D4'

fig = plt.figure(figsize=(180*MM, 132*MM))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.25], hspace=.22, wspace=.14)

def box(ax, x, y, w, h, text, fc, ec='#555555', fs=6.6, lw=.8, weight=None):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                 boxstyle='round,pad=0.006,rounding_size=0.015',
                 facecolor=fc, edgecolor=ec, linewidth=lw, zorder=2))
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=fs,
            zorder=3, linespacing=1.4, fontweight=weight)

def arrow(ax, x1, y1, x2, y2, lw=.9, col='#666666'):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>',
                 mutation_scale=7, linewidth=lw, color=col, zorder=1))

def blank(ax):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

def letter(ax, l, x=-.01, y=1.02):
    ax.text(x, y, l, transform=ax.transAxes, fontsize=10,
            fontweight='bold', va='top', ha='left')

# ================================================== a: cohorts
ax = fig.add_subplot(gs[0, 0]); blank(ax); letter(ax, 'a')
ax.text(.5, .99, 'Cohort construction', ha='center', va='top',
        fontsize=8.2, fontweight='bold')

box(ax, .13, .800, .74, .100,
    'UCL/Ibadan thin blood films\n149 slides \u00b7 73 SCD+ / 76 SCD\u2212', '#F2F2F2')

arrow(ax, .40, .800, .27, .720)
arrow(ax, .60, .800, .73, .720)

ax.text(.25, .700, 'Primary cohort', ha='center', fontsize=6.8,
        fontweight='bold', color=POS)
ax.text(.75, .700, 'Batch-matched cohort', ha='center', fontsize=6.8,
        fontweight='bold', color=NEG)

box(ax, .02, .500, .46, .165,
    'date 280120\n25 slides, all SCD+', '#FBE3D6', ec=POS)
box(ax, .52, .500, .46, .165,
    '080119: 7+ / 4\u2212\n101017: 8+ / 5\u2212', '#DCE9F5', ec=NEG)

arrow(ax, .25, .500, .25, .430)
arrow(ax, .75, .500, .75, .430)

box(ax, .02, .285, .46, .145,
    'one field per slide\n3,762 cells\nSections 3.1\u20133.8', '#DFF2EA', ec=GTC, fs=6.2)
box(ax, .52, .285, .46, .145,
    'one field per slide\n4,070 cells\nSection 3.9', '#DFF2EA', ec=GTC, fs=6.2)

ax.text(.5, .135, 'Within a batch, positive and negative slides share\n'
        'stain, scanner and session \u2014 controlled by construction',
        ha='center', va='center', fontsize=6.2, style='italic',
        color='#555555', linespacing=1.5)

# ================================================== b: ground truth
ax = fig.add_subplot(gs[0, 1]); blank(ax); letter(ax, 'b')
ax.text(.5, .99, 'Ground-truth generation', ha='center', va='top',
        fontsize=8.2, fontweight='bold')

steps = [
    ('Field image   2560 \u00d7 2160 px', '#F2F2F2', 0),
    ('SAM ViT-B automatic masks\n32 pts/side \u00b7 IoU 0.86 \u00b7 stability 0.92',
     '#F2F2F2', 0),
    ('Reject masks with < 50% overlap\nwith Otsu foreground', '#FFF3DC', 1),
    ('Containment NMS at 0.35\nremoves nested proposals', '#FFF3DC', 1),
    ('Area window 4,000\u201330,000 px', '#F2F2F2', 0),
    ('Instance ground truth', '#DFF2EA', 0)]

y, h, gap = .845, .097, .033
clean_top = clean_bot = None
for i, (t, c, tag) in enumerate(steps):
    ec = SEED if tag else '#555555'
    box(ax, .08, y, .76, h, t, c, ec=ec, lw=1.1 if tag else .8)
    if tag:
        if clean_top is None:
            clean_top = y + h
        clean_bot = y
    if i < len(steps)-1:
        arrow(ax, .46, y, .46, y-gap)
    y -= (h+gap)

ax.plot([.865, .895, .895, .865], [clean_top, clean_top, clean_bot, clean_bot],
        color=SEED, lw=1.0, clip_on=False)
ax.text(.945, (clean_top+clean_bot)/2, 'cleaning steps\nadded here', fontsize=5.9,
        color=SEED, rotation=90, va='center', ha='center', linespacing=1.3)

ax.text(.46, .085, 'Nested proposals otherwise double-counted\n'
        '12.1% of foreground pixels',
        ha='center', va='center', fontsize=6.2, style='italic',
        color='#555555', linespacing=1.5)

# ================================================== c: methods
ax = fig.add_subplot(gs[1, :]); blank(ax); letter(ax, 'c')
ax.text(.5, 1.00, 'Segmentation methods \u2014 identical except at the seeding step',
        ha='center', va='top', fontsize=8.2, fontweight='bold')

ax.text(.245, .885, 'Shared front end', ha='center', fontsize=6.8,
        color='#777777', style='italic')
fx, fw = .015, .105
for i, t in enumerate(['Gaussian\nblur', 'Otsu\nthreshold',
                       'Morphological\nopening', 'Distance\ntransform']):
    box(ax, fx, .735, fw, .125, t, SHARED, fs=6.2)
    if i < 3:
        arrow(ax, fx+fw, .797, fx+fw+.013, .797)
    fx += fw + .013
arrow(ax, .487, .797, .512, .797)

ax.plot([.505, .505], [.075, .93], color=SEED, lw=1.1, ls=(0, (4, 2)), zorder=0)
ax.text(.505, .900, 'divergence', ha='center', fontsize=6.0,
        color=SEED, bbox=dict(boxstyle='round,pad=.15', fc='white',
        ec='none'))

rules = [
    ('original', 'seed where  dist > 0.3 \u00d7 field max', POS, .620),
    ('fix$_A$', 'same seeds; watershed instances kept\nrather than re-labelled',
     '#E69F00', .485),
    ('fix$_B$', 'seed where  dist > 0.3 \u00d7 local max\nper connected component',
     NEG, .350),
    ('fix$_C$', 'seed at extended maxima, depth $h$\n($h$ swept 3\u201312)',
     '#009E73', .215)]

for name, rule, col, yy in rules:
    box(ax, .522, yy, .098, .110, name, 'white', ec=col, lw=1.4, fs=6.6,
        weight='bold')
    box(ax, .636, yy, .238, .110, rule, 'white', ec='#C4C4C4', fs=5.3)
    arrow(ax, .622, yy+.055, .632, yy+.055, col=col)
    arrow(ax, .876, yy+.055, .884, yy+.055, col='#999999')

box(ax, .522, .075, .352, .100,
    'Cellpose-SAM\nno distance transform', '#EFEFEF', fs=5.8)

box(ax, .882, .215, .080, .515,
    'Area\nfilter\n\n[5,000,\n17,000)\npx', SHARED, fs=5.8)
arrow(ax, .922, .215, .922, .185)
box(ax, .882, .075, .080, .100, 'Predicted\ninstances', '#DFF2EA', fs=5.6)

ax.text(.245, .625, 'Evaluation', ha='center', fontsize=7.6, fontweight='bold')
yy = .505
for e in ['IoU 0.3 / 0.5 / 0.7 / 0.75',
          'stratified by contact fraction',
          'stratified by eccentricity',
          'per-slide, by phenotype']:
    box(ax, .015, yy, .460, .078, e, EVAL, fs=6.2, ec='#AAAAAA')
    yy -= .092

ax.text(.245, .105, 'Recall \u00b7 precision \u00b7 F1 \u00b7 failure mode\n'
        'attribute-conditional recall \u00b7 class disparity',
        ha='center', va='center', fontsize=6.2, style='italic',
        color='#555555', linespacing=1.5)

for ext in ('pdf', 'png', 'tiff'):
    kw = dict(pil_kwargs={'compression': 'tiff_lzw'}) if ext == 'tiff' else {}
    fig.savefig(OUT / f'fig0_methods_overview.{ext}', **kw)
plt.close(fig)
print('saved fig0_methods_overview')

saved fig0_methods_overview


In [2]:
"""
FIG 1 - STUDY DESIGN AND METHODS OVERVIEW
(a) cohort construction  (b) ground-truth generation
(c) the five segmenters, identical except at the seeding step
"""
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pathlib import Path

OUT = Path('/kaggle/working/figures'); OUT.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 600,
    'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 7.5, 'savefig.bbox': 'tight', 'savefig.pad_inches': .03,
    'pdf.fonttype': 42, 'ps.fonttype': 42})

MM = 1/25.4
POS, NEG = '#D55E00', '#0072B2'
GTC, SHARED = '#009E73', '#E8E8E8'
SEED, EVAL = '#E69F00', '#D4D4D4'

fig = plt.figure(figsize=(180*MM, 132*MM))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.25], hspace=.22, wspace=.14)

def box(ax, x, y, w, h, text, fc, ec='#555555', fs=6.6, lw=.8, weight=None):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                 boxstyle='round,pad=0,rounding_size=0.012',
                 facecolor=fc, edgecolor=ec, linewidth=lw, zorder=2,
                 mutation_aspect=0.35))
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=fs,
            zorder=3, linespacing=1.4, fontweight=weight)

def arrow(ax, x1, y1, x2, y2, lw=.9, col='#666666'):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>',
                 mutation_scale=7, linewidth=lw, color=col, zorder=1))

def blank(ax):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')

def letter(ax, l, x=-.01, y=1.02):
    ax.text(x, y, l, transform=ax.transAxes, fontsize=10,
            fontweight='bold', va='top', ha='left')

# ================================================== a: cohorts
ax = fig.add_subplot(gs[0, 0]); blank(ax); letter(ax, 'a')
ax.text(.5, .99, 'Cohort construction', ha='center', va='top',
        fontsize=8.2, fontweight='bold')

box(ax, .13, .800, .74, .100,
    'UCL/Ibadan thin blood films\n149 slides \u00b7 73 SCD+ / 76 SCD\u2212',
    '#F2F2F2', fs=6.4)

arrow(ax, .40, .800, .27, .720)
arrow(ax, .60, .800, .73, .720)

ax.text(.25, .700, 'Primary cohort', ha='center', fontsize=6.8,
        fontweight='bold', color=POS)
ax.text(.75, .700, 'Batch-matched cohort', ha='center', fontsize=6.8,
        fontweight='bold', color=NEG)

box(ax, .02, .500, .46, .165,
    'date 280120\n25 slides, all SCD+', '#FBE3D6', ec=POS)
box(ax, .52, .500, .46, .165,
    '080119: 7+ / 4\u2212\n101017: 8+ / 5\u2212', '#DCE9F5', ec=NEG)

arrow(ax, .25, .500, .25, .430)
arrow(ax, .75, .500, .75, .430)

box(ax, .01, .285, .48, .145,
    'one field per slide\n3,762 cells\nSections 3.1\u20133.8', '#DFF2EA', ec=GTC, fs=6.2)
box(ax, .51, .285, .48, .145,
    'one field per slide\n4,070 cells\nSection 3.9', '#DFF2EA', ec=GTC, fs=6.2)

ax.text(.5, .135, 'Within a batch, positive and negative slides share\n'
        'stain, scanner and session \u2014 controlled by construction',
        ha='center', va='center', fontsize=6.2, style='italic',
        color='#555555', linespacing=1.5)

# ================================================== b: ground truth
ax = fig.add_subplot(gs[0, 1]); blank(ax); letter(ax, 'b')
ax.text(.5, .99, 'Ground-truth generation', ha='center', va='top',
        fontsize=8.2, fontweight='bold')

steps = [
    ('Field image   2560 \u00d7 2160 px', '#F2F2F2', 0),
    ('SAM ViT-B automatic masks\n32 pts/side, IoU 0.86, stability 0.92',
     '#F2F2F2', 0),
    ('Reject masks < 50% overlapping\nthe Otsu foreground', '#FFF3DC', 1),
    ('Containment NMS at 0.35\nremoves nested proposals', '#FFF3DC', 1),
    ('Area window 4,000\u201330,000 px', '#F2F2F2', 0),
    ('Instance ground truth', '#DFF2EA', 0)]

y, h, gap = .845, .097, .033
clean_top = clean_bot = None
for i, (t, c, tag) in enumerate(steps):
    ec = SEED if tag else '#555555'
    box(ax, .06, y, .80, h, t, c, ec=ec, lw=1.1 if tag else .8, fs=6.4)
    if tag:
        if clean_top is None:
            clean_top = y + h
        clean_bot = y
    if i < len(steps)-1:
        arrow(ax, .46, y, .46, y-gap)
    y -= (h+gap)

ax.plot([.865, .895, .895, .865], [clean_top, clean_top, clean_bot, clean_bot],
        color=SEED, lw=1.0, clip_on=False)
ax.text(.945, (clean_top+clean_bot)/2, 'cleaning steps\nadded here', fontsize=5.9,
        color=SEED, rotation=90, va='center', ha='center', linespacing=1.3)

ax.text(.46, .085, 'Nested proposals otherwise double-counted\n'
        '12.1% of foreground pixels',
        ha='center', va='center', fontsize=6.2, style='italic',
        color='#555555', linespacing=1.5)

# ================================================== c: methods
ax = fig.add_subplot(gs[1, :]); blank(ax); letter(ax, 'c')
ax.text(.5, 1.00, 'Segmentation methods \u2014 identical except at the seeding step',
        ha='center', va='top', fontsize=8.4, fontweight='bold')

# shared front end, running down the left
ax.text(.105, .855, 'Shared front end', ha='center', fontsize=7.0,
        color='#777777', style='italic')
fy, fh = .690, .105
for i, t in enumerate(['Gaussian blur', 'Otsu threshold',
                       'Morphological opening', 'Distance transform']):
    box(ax, .015, fy, .185, fh, t, SHARED, fs=6.2)
    if i < 3:
        arrow(ax, .105, fy, .105, fy - .038)
    fy -= (fh + .038)

arrow(ax, .195, .300, .240, .300)
arrow(ax, .240, .300, .240, .620)
arrow(ax, .240, .620, .268, .620)

# divergence line
ax.plot([.262, .262], [.075, .800], color=SEED, lw=1.2, ls=(0, (4, 2)), zorder=0)
ax.text(.262, .845, 'divergence', ha='center', fontsize=6.6, color=SEED,
        bbox=dict(boxstyle='round,pad=.18', fc='white', ec='none'))

rules = [
    ('original', 'seed where  dist > 0.3 \u00d7 field max', POS, .620),
    ('fix$_A$', 'same seeds; watershed instances kept, not re-labelled',
     '#E69F00', .485),
    ('fix$_B$', 'seed where  dist > 0.3 \u00d7 local max (per component)',
     NEG, .350),
    ('fix$_C$', 'seed at extended maxima of depth $h$  ($h$ swept 3\u201312)',
     '#009E73', .215)]

for name, rule, col, yy in rules:
    box(ax, .275, yy, .098, .110, name, 'white', ec=col, lw=1.5, fs=7.0,
        weight='bold')
    box(ax, .400, yy, .428, .110, rule, 'white', ec='#C4C4C4', fs=6.2)
    arrow(ax, .376, yy+.055, .396, yy+.055, col=col)
    arrow(ax, .828, yy+.055, .846, yy+.055, col='#999999')

box(ax, .275, .075, .553, .100,
    'Cellpose-SAM   \u00b7   no distance transform, contemporary reference',
    '#EFEFEF', fs=6.2)
arrow(ax, .828, .125, .846, .125, col='#999999')

box(ax, .848, .215, .080, .515,
    'Area\nfilter\n\n[5,000,\n17,000)\npx', SHARED, fs=6.0)
arrow(ax, .888, .215, .888, .182)
box(ax, .848, .075, .080, .100, 'Predicted\ninstances', '#DFF2EA', fs=6.0)

ax.text(.60, .015, 'All methods evaluated at IoU 0.3\u20130.75, stratified by '
        'contact fraction, eccentricity and phenotype',
        ha='center', va='center', fontsize=6.2, style='italic',
        color='#555555')

for ext in ('pdf', 'png', 'tiff'):
    kw = dict(pil_kwargs={'compression': 'tiff_lzw'}) if ext == 'tiff' else {}
    fig.savefig(OUT / f'fig0_methods_overview.{ext}', **kw)
plt.close(fig)
print('saved fig0_methods_overview')

saved fig0_methods_overview


In [1]:
"""
FIG 1 - STUDY DESIGN AND METHODS OVERVIEW
Boxes are measured from the rendered text, so labels cannot overflow.
"""
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from pathlib import Path

OUT = Path('/kaggle/working/figures'); OUT.mkdir(parents=True, exist_ok=True)

mpl.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 600,
    'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 7.5, 'savefig.bbox': 'tight', 'savefig.pad_inches': .04,
    'pdf.fonttype': 42, 'ps.fonttype': 42})

MM = 1/25.4
POS, NEG, GTC = '#D55E00', '#0072B2', '#009E73'
SHARED, SEED = '#EAEAEA', '#E69F00'

fig = plt.figure(figsize=(180*MM, 138*MM))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1.18], hspace=.26, wspace=.16)

# ---------------------------------------------------------------- helpers
def blank(ax):
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis('off')
    return ax

def letter(ax, l):
    ax.text(-.02, 1.04, l, transform=ax.transAxes, fontsize=10,
            fontweight='bold', va='top', ha='left')

def measure(ax, text, fs, weight=None):
    """Text bounding box in axes coordinates, without leaving it drawn."""
    t = ax.text(.5, .5, text, ha='center', va='center', fontsize=fs,
                linespacing=1.45, fontweight=weight, alpha=0)
    fig.canvas.draw()
    bb = t.get_window_extent()
    inv = ax.transAxes.inverted()
    (x0, y0), (x1, y1) = inv.transform([[bb.x0, bb.y0], [bb.x1, bb.y1]])
    t.remove()
    return x1 - x0, y1 - y0

def drawbox(ax, cx, cy, w, h, text, fc, ec='#555555', fs=6.4, lw=.9,
            weight=None, tcol='black'):
    ax.add_patch(FancyBboxPatch((cx - w/2, cy - h/2), w, h,
                 boxstyle='round,pad=0,rounding_size=0.010',
                 facecolor=fc, edgecolor=ec, linewidth=lw, zorder=2,
                 mutation_aspect=.35))
    ax.text(cx, cy, text, ha='center', va='center', fontsize=fs, zorder=3,
            linespacing=1.45, fontweight=weight, color=tcol)

def stack(ax, cx, top, items, fs=6.4, padx=.055, pady=.045, gap=.028,
          arrows=True):
    """Vertical stack of boxes, all the width of the widest text."""
    sizes = [measure(ax, t, fs) for t, *_ in items]
    W = max(w for w, _ in sizes) + padx
    y = top; placed = []
    for (item, size) in zip(items, sizes):
        t, fc = item[0], item[1]
        ec = item[2] if len(item) > 2 else '#555555'
        lw = item[3] if len(item) > 3 else .9
        h = size[1] + pady
        cy = y - h/2
        drawbox(ax, cx, cy, W, h, t, fc, ec=ec, fs=fs, lw=lw)
        placed.append((cy, h))
        if arrows and item is not items[-1]:
            arrow(ax, cx, cy - h/2, cx, cy - h/2 - gap)
        y -= (h + gap)
    return W, placed, y

def arrow(ax, x1, y1, x2, y2, col='#666666', lw=.9):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle='-|>',
                 mutation_scale=7, linewidth=lw, color=col, zorder=1))

def note(ax, cx, cy, text, fs=6.2):
    ax.text(cx, cy, text, ha='center', va='center', fontsize=fs,
            style='italic', color='#555555', linespacing=1.5)

# ================================================== a: cohorts
ax = blank(fig.add_subplot(gs[0, 0])); letter(ax, 'a')
ax.text(.5, 1.00, 'Cohort construction', ha='center', va='top',
        fontsize=8.4, fontweight='bold')

w, h = measure(ax, 'UCL/Ibadan thin blood films\n'
               '149 slides \u00b7 73 SCD+ / 76 SCD\u2212', 6.4)
drawbox(ax, .5, .845, w + .06, h + .045,
        'UCL/Ibadan thin blood films\n149 slides \u00b7 73 SCD+ / 76 SCD\u2212',
        '#F2F2F2')

arrow(ax, .42, .845 - (h+.045)/2, .28, .730)
arrow(ax, .58, .845 - (h+.045)/2, .72, .730)

for cx, lab, col in [(.25, 'Primary cohort', POS),
                     (.75, 'Batch-matched cohort', NEG)]:
    ax.text(cx, .700, lab, ha='center', va='center', fontsize=7.0,
            fontweight='bold', color=col)

left = [('date 280120\n25 slides, all SCD+', '#FBE3D6', POS, 1.2),
        ('one field per slide\n3,762 cells\nSections 3.1\u20133.8',
         '#DFF2EA', GTC, 1.2)]
right = [('080119: 7+ / 4\u2212\n101017: 8+ / 5\u2212', '#DCE9F5', NEG, 1.2),
         ('one field per slide\n4,070 cells\nSection 3.9',
          '#DFF2EA', GTC, 1.2)]
_, _, ya = stack(ax, .25, .655, left, fs=6.4, gap=.050)
stack(ax, .75, .655, right, fs=6.4, gap=.050)

note(ax, .5, ya - .085, 'Within a batch, positive and negative slides share\n'
     'stain, scanner and session \u2014 controlled by construction')

# ================================================== b: ground truth
ax = blank(fig.add_subplot(gs[0, 1])); letter(ax, 'b')
ax.text(.5, 1.00, 'Ground-truth generation', ha='center', va='top',
        fontsize=8.4, fontweight='bold')

steps = [
    ('Field image  2560 \u00d7 2160 px', '#F2F2F2'),
    ('SAM ViT-B automatic masks\n32 pts/side, IoU 0.86, stability 0.92', '#F2F2F2'),
    ('Reject masks < 50% overlapping\nthe Otsu foreground', '#FFF3DC', SEED, 1.3),
    ('Containment NMS at 0.35\nremoves nested proposals', '#FFF3DC', SEED, 1.3),
    ('Area window 4,000\u201330,000 px', '#F2F2F2'),
    ('Instance ground truth', '#DFF2EA', GTC, 1.2)]

W, placed, yb = stack(ax, .46, .900, steps, fs=6.1, pady=.038, gap=.020)

ct = placed[2][0] + placed[2][1]/2
cb = placed[3][0] - placed[3][1]/2
xb = .46 + W/2 + .02
ax.plot([xb, xb+.03, xb+.03, xb], [ct, ct, cb, cb], color=SEED, lw=1.0)
ax.text(xb+.085, (ct+cb)/2, 'cleaning steps\nadded here', fontsize=5.9,
        color=SEED, rotation=90, va='center', ha='center', linespacing=1.3)

note(ax, .46, yb - .075, 'Nested proposals otherwise double-counted\n'
     '12.1% of foreground pixels')

# ================================================== c: methods
ax = blank(fig.add_subplot(gs[1, :])); letter(ax, 'c')
ax.text(.5, 1.02, 'Segmentation methods \u2014 identical except at the seeding step',
        ha='center', va='top', fontsize=8.4, fontweight='bold')

ax.text(.100, .900, 'Shared front end', ha='center', va='top', fontsize=7.0,
        color='#777777', style='italic')
front = [(t, SHARED) for t in ['Gaussian blur', 'Otsu threshold',
                               'Morphological opening', 'Distance transform']]
Wf, pf, _ = stack(ax, .115, .830, front, fs=6.4, padx=.035, pady=.055, gap=.045)

xin = .115 + Wf/2
xmid = xin + .020
arrow(ax, xin, pf[-1][0], xmid, pf[-1][0])
arrow(ax, xmid, pf[-1][0], xmid, .640)
arrow(ax, xmid, .640, .240, .640)

ax.plot([.232, .232], [.045, .820], color=SEED, lw=1.2, ls=(0, (4, 2)), zorder=0)
ax.text(.244, .845, 'divergence', ha='left', va='bottom', fontsize=6.6,
        color=SEED)

rules = [('original', 'seed where  dist > 0.3 \u00d7 field max', POS),
         ('fix$_A$', 'same seeds; watershed instances kept, not re-labelled',
          '#E69F00'),
         ('fix$_B$', 'seed where  dist > 0.3 \u00d7 local max, per component', NEG),
         ('fix$_C$', 'seed at extended maxima of depth $h$  ($h$ swept 3\u201312)',
          GTC)]

nw = max(measure(ax, n, 7.0, 'bold')[0] for n, _, _ in rules) + .045
rw = max(measure(ax, r, 6.4)[0] for _, r, _ in rules) + .045
nx = .245 + nw/2
rx = nx + nw/2 + .022 + rw/2

ys = [.640, .500, .360, .220]
for (name, rule, col), yy in zip(rules, ys):
    hh = measure(ax, name, 7.0, 'bold')[1] + .060
    drawbox(ax, nx, yy, nw, hh, name, 'white', ec=col, lw=1.5, fs=7.0,
            weight='bold')
    drawbox(ax, rx, yy, rw, hh, rule, 'white', ec='#C8C8C8', fs=6.4)
    arrow(ax, nx + nw/2, yy, rx - rw/2, yy, col=col)
    arrow(ax, rx + rw/2, yy, rx + rw/2 + .026, yy, col='#999999')

cp = 'Cellpose-SAM   \u00b7   no distance transform, contemporary reference'
ch = measure(ax, cp, 6.4)[1] + .060
drawbox(ax, (nx - nw/2 + rx + rw/2)/2, .095, rx + rw/2 - nx + nw/2, ch,
        cp, '#EFEFEF', fs=6.4)
arrow(ax, rx + rw/2, .095, rx + rw/2 + .026, .095, col='#999999')

bx = rx + rw/2 + .026 + .048
af = 'Area\nfilter\n\n[5,000,\n17,000)\npx'
aw = measure(ax, af, 6.2)[0] + .040
drawbox(ax, bx, .430, aw, .560, af, SHARED, fs=6.2)
arrow(ax, bx, .150, bx, .118)
pi = 'Predicted\ninstances'
ph = measure(ax, pi, 6.2)[1] + .050
drawbox(ax, bx, .095 - .002, aw, ph, pi, '#DFF2EA', ec=GTC, fs=6.2, lw=1.2)

note(ax, .5, -.005, 'All methods evaluated at IoU 0.3\u20130.75, stratified by '
     'contact fraction, eccentricity and phenotype')

for ext in ('pdf', 'png', 'tiff'):
    kw = dict(pil_kwargs={'compression': 'tiff_lzw'}) if ext == 'tiff' else {}
    fig.savefig(OUT / f'fig0_methods_overview.{ext}', **kw)
plt.close(fig)
print('saved fig0_methods_overview')

saved fig0_methods_overview


In [2]:
#!/usr/bin/env python3
"""
Paper 2 - Figure 1 (Section 1, Introduction)
Conceptual figure: (a) where the audit sits in the published pipeline,
(b) the measurement-validity principle, (c) preview of the three findings.

Conceptual only - panel (b) is schematic and says so. All numeric values
shown are measured (Sections 3.1, 3.7 and the batch-matched analysis).

Outputs PDF + PNG + TIFF at 600 dpi, Okabe-Ito palette, Type 42 fonts.
"""
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Ellipse, Circle, FancyArrowPatch
import numpy as np

FONT = "DejaVu Sans"   # <- swap to "Arial"/"Helvetica" to match figs 2-12
matplotlib.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42,
    "font.family": "sans-serif", "font.sans-serif": [FONT],
})

OI = {"blue": "#0072B2", "verm": "#D55E00", "green": "#009E73",
      "orange": "#E69F00", "purple": "#CC79A7", "skyblue": "#56B4E9"}
GREY, PANEL, WARM = "#9A9A9A", "#F4F4F4", "#FDF3EC"
FS_B, FS_L, FS_T, FS_P = 6.4, 7.2, 8.0, 10.0

fig = plt.figure(figsize=(7.2, 6.5))
gs = fig.add_gridspec(3, 1, height_ratios=[1.00, 1.28, 0.50], hspace=0.10,
                      left=0.015, right=0.985, top=0.97, bottom=0.015)
axA, axB, axC = (fig.add_subplot(gs[i]) for i in range(3))
for ax in (axA, axB, axC):
    ax.set_xlim(0, 100); ax.set_ylim(0, 100); ax.axis("off")


def box(ax, x, y, w, h, text, fc="white", ec=GREY, lw=0.9, fs=FS_B,
        tc="black", bold=False, ls="-"):
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                 boxstyle="round,pad=0,rounding_size=1.6", facecolor=fc,
                 edgecolor=ec, linewidth=lw, linestyle=ls, zorder=2))
    ax.text(x + w/2, y + h/2, text, ha="center", va="center", fontsize=fs,
            color=tc, zorder=3, linespacing=1.5,
            fontweight="bold" if bold else "normal")


def arrow(ax, x0, y0, x1, y1, color="black", lw=1.0, ls="-", ms=8):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>",
                 mutation_scale=ms, linewidth=lw, color=color, linestyle=ls,
                 shrinkA=0, shrinkB=0, zorder=4))


# ============================================================== PANEL a
axA.text(-1.0, 91, "a", fontsize=FS_P, fontweight="bold", va="center")
axA.text(3.5, 91, "The published pipeline. Only the classifier stage has been "
                "evaluated in its own right.", fontsize=FS_L, va="center")

BW, GAP, BY, BH = 17.6, 2.8, 50, 30
xs = [i * (BW + GAP) for i in range(5)]
stages = [
    ("Digitised thin film\n2560 $\\times$ 2160 px\nEDoF projection", "white", GREY, 0.9),
    ("rbc_segmentation()\nOtsu $\\rightarrow$ distance transform\n$\\rightarrow$ seed at 0.3 $\\times$ max\n$\\rightarrow$ watershed $\\rightarrow$ area filter",
     WARM, OI["verm"], 1.5),
    ("Single-cell crops\nfrom segmentation\nbounding boxes", "white", GREY, 0.9),
    ("CNN classifier\nabnormal vs\nnormal RBC", "#E8F4FA", OI["blue"], 1.5),
    ("Slide-level\nsickle fraction\n$\\rightarrow$ diagnosis", "white", GREY, 0.9),
]
for x, (t, fc, ec, lw) in zip(xs, stages):
    box(axA, x, BY, BW, BH, t, fc=fc, ec=ec, lw=lw, fs=5.8)
for i in range(4):
    arrow(axA, xs[i] + BW + 0.3, BY + BH/2, xs[i+1] - 0.3, BY + BH/2)

box(axA, xs[1], 20, BW, 20,
    "no instance-level\nevaluation published\n\u2014 audited here",
    fc="white", ec=OI["verm"], lw=1.0, ls=(0, (3, 2)), tc=OI["verm"], fs=6.1)
arrow(axA, xs[1] + BW/2, BY - 0.3, xs[1] + BW/2, 40.3, color=OI["verm"],
      lw=0.9, ls=(0, (3, 2)))
box(axA, xs[3], 20, BW, 20, "reported\nprecision 0.86\nrecall 0.69",
    fc="white", ec=OI["blue"], lw=1.0, ls=(0, (3, 2)), tc=OI["blue"], fs=6.1)
arrow(axA, xs[3] + BW/2, BY - 0.3, xs[3] + BW/2, 40.3, color=OI["blue"],
      lw=0.9, ls=(0, (3, 2)))

axA.annotate("", xy=(xs[4] + BW, 12), xytext=(xs[1], 12),
             arrowprops=dict(arrowstyle="-", lw=0.8, color="black"))
for xx in (xs[1], xs[4] + BW):
    axA.plot([xx, xx], [12, 15], lw=0.8, color="black")
axA.text((xs[1] + xs[4] + BW)/2, 6.5,
         "errors here propagate to everything downstream",
         ha="center", va="center", fontsize=FS_L, style="italic")

# ============================================================== PANEL b
axB.text(-1.0, 92, "b", fontsize=FS_P, fontweight="bold", va="center")
axB.text(3.5, 92, "Why aggregate recall is not enough. A conservative segmenter "
                "can be almost perfectly precise and still return a biased sample.",
         fontsize=FS_L, va="center")

rng = np.random.default_rng(11)
NC, NR = 6, 4
ELONG = {1, 4, 7, 10, 13, 16, 19, 22}          # 8 of 24
LOST  = {1, 4, 7, 13, 19, 3, 11, 20}           # 5 elongated + 3 round


def grid(ax, x0, y0, lost=frozenset(), seed=0):
    rr = np.random.default_rng(seed)
    for j in range(NR):
        for i in range(NC):
            k = j * NC + i
            cx, cy = x0 + 3.6 + i * 5.6, y0 - j * 6.4
            el, gone = k in ELONG, k in lost
            if gone:
                fc, ec, lw, ls, z = "none", GREY, 0.7, (0, (1.5, 1.3)), 2
            else:
                c = OI["verm"] if el else OI["blue"]
                fc, ec, lw, ls, z = c, c, 0.6, "-", 3
            if el:
                p = Ellipse((cx, cy), 5.4, 2.1, angle=float(rr.uniform(0, 180)),
                            facecolor=fc, edgecolor=ec, linewidth=lw,
                            linestyle=ls, zorder=z)
            else:
                p = Circle((cx, cy), 1.7, facecolor=fc, edgecolor=ec,
                           linewidth=lw, linestyle=ls, zorder=z)
            ax.add_patch(p)


axB.add_patch(FancyBboxPatch((0.5, 22), 36, 60,
              boxstyle="round,pad=0,rounding_size=1.6", facecolor=PANEL,
              edgecolor="none", zorder=0))
axB.text(18.5, 77, "True cell population", ha="center", fontsize=FS_T,
         fontweight="bold")
grid(axB, 2.0, 68, seed=3)

axB.add_patch(FancyBboxPatch((63.5, 22), 36, 60,
              boxstyle="round,pad=0,rounding_size=1.6", facecolor=WARM,
              edgecolor=OI["verm"], linewidth=1.0, zorder=0))
axB.text(81.5, 77, "Cells the pipeline returns", ha="center", fontsize=FS_T,
         fontweight="bold", color=OI["verm"])
grid(axB, 65.0, 68, lost=LOST, seed=3)

box(axB, 41.5, 47, 17, 20,
    "seed where\ndist $>$ 0.3 $\\times$\nfield maximum", fc="white",
    ec=OI["verm"], lw=1.2, fs=6.3)
arrow(axB, 37.2, 57, 41.2, 57, lw=1.1)
arrow(axB, 58.8, 57, 63.0, 57, lw=1.1)

axB.text(18.5, 27.5, "every cell present", ha="center", fontsize=FS_B,
         style="italic", color="#444444")
axB.text(81.5, 30.5, "elongated cells lost preferentially",
         ha="center", fontsize=FS_B, style="italic", color=OI["verm"])
axB.text(81.5, 25.5, "high precision \u2014 but the wrong sample",
         ha="center", fontsize=FS_B, style="italic", color=OI["verm"])

axB.text(99.0, 18.5, "panel b is schematic; all values in a and c are measured",
         ha="right", va="center", fontsize=5.8, color=GREY, style="italic")

lx, ly = 2.0, 12.0
axB.add_patch(Circle((lx + 1.0, ly), 1.7, facecolor=OI["blue"],
              edgecolor=OI["blue"], lw=0.6))
axB.text(lx + 3.4, ly, "round erythrocyte", va="center", fontsize=FS_B)
axB.add_patch(Ellipse((lx + 26.0, ly), 5.4, 2.1, angle=25,
              facecolor=OI["verm"], edgecolor=OI["verm"], lw=0.6))
axB.text(lx + 30.0, ly, "elongated (sickled) erythrocyte \u2014 the diagnostic feature",
         va="center", fontsize=FS_B)
axB.add_patch(Circle((lx + 1.0, ly - 6.5), 1.7, facecolor="none",
              edgecolor=GREY, lw=0.7, linestyle=(0, (1.5, 1.3))))
axB.text(lx + 3.4, ly - 6.5, "cell discarded by the segmenter", va="center",
         fontsize=FS_B)

# ============================================================== PANEL c
axC.text(-1.0, 88, "c", fontsize=FS_P, fontweight="bold", va="center")
axC.text(3.5, 88, "What the audit finds", fontsize=FS_L, fontweight="bold",
         va="center")

CW, CG = 31.6, 2.6
cx0 = [i * (CW + CG) for i in range(3)]
cards = [
    ("55.7%", "of erythrocytes recovered\nat 97.9% precision\n(3,762 cells, 25 fields)", OI["blue"]),
    ("$-$17%", "relative under-count of the\nsickle fraction: 30.9% true\nvs 25.6% reported", OI["orange"]),
    ("18.8 pp", "lower recall on SCD-positive\nthan SCD-negative films\n($P$ = 0.020, 24 slides)", OI["verm"]),
]
for x, (big, small, col) in zip(cx0, cards):
    axC.add_patch(FancyBboxPatch((x, 8), CW, 68,
                  boxstyle="round,pad=0,rounding_size=1.6", facecolor="white",
                  edgecolor=col, linewidth=1.3, zorder=2))
    axC.text(x + CW/2, 60, big, ha="center", va="center", fontsize=13,
             fontweight="bold", color=col, zorder=3)
    axC.text(x + CW/2, 28, small, ha="center", va="center", fontsize=FS_B,
             color="#222222", linespacing=1.55, zorder=3)

OUT = "/mnt/user-data/outputs"
os.makedirs(OUT, exist_ok=True)
stem = os.path.join(OUT, "fig1_audit_rationale")
for ext, kw in [(".pdf", {}), (".png", {}),
                (".tiff", {"pil_kwargs": {"compression": "tiff_lzw"}})]:
    fig.savefig(stem + ext, dpi=600, bbox_inches="tight", facecolor="white", **kw)
print("wrote", stem)

wrote /mnt/user-data/outputs/fig1_audit_rationale


In [6]:
from pathlib import Path
for p in sorted(Path('/kaggle/input/paper2-gt-and-results').rglob('*'))[:60]:
    print(p.relative_to('/kaggle/input/paper2-gt-and-results'),
          f'{p.stat().st_size/1e6:.1f} MB' if p.is_file() else '')

In [1]:
import numpy as np, cv2, pandas as pd, glob, os, zipfile
from pathlib import Path
from scipy.ndimage import binary_fill_holes
from skimage.morphology import h_maxima
from skimage.measure import label as sklabel
from skimage.segmentation import watershed as skws

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/thin_films_part1/sickle3-edofed')
MIN_SIZE, MAX_SIZE, TAU, H = 5000, 17000, 0.5, 9

# locate ground truth (primary cohort) and shape table
GT = Path('/kaggle/working/gt_masks'); GT.mkdir(exist_ok=True)
if not list(GT.glob('280120-*.npy')):
    z = [p for p in glob.glob('/kaggle/input/**/gt_masks.zip', recursive=True)]
    npys = [p for p in glob.glob('/kaggle/input/**/280120-*.npy', recursive=True)]
    if z: zipfile.ZipFile(z[0]).extractall(GT)
    elif npys: GT = Path(npys[0]).parent
gts = sorted(GT.rglob('280120-*.npy')); print(len(gts), 'gt fields from', GT)
shp = pd.read_csv(glob.glob('/kaggle/input/**/gt_shape.csv', recursive=True)[0])

def _front(im):
    g = cv2.GaussianBlur(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, th = cv2.threshold(g, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = np.ones((3, 3), np.uint8)
    op = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=2)
    return op, cv2.dilate(op, k, iterations=2), cv2.distanceTransform(op, cv2.DIST_L2, 5)

def seg_fix_c(im, h):
    op, bg, dist = _front(im); fg = op > 0
    seeds = sklabel(h_maxima(dist, h) > 0)
    if seeds.max() == 0: return np.zeros(fg.shape, np.int32)
    lab = skws(-dist, seeds, mask=fg); out = np.zeros(lab.shape, np.int32); nx = 0
    for i in range(1, lab.max() + 1):
        r = binary_fill_holes(lab == i); a = int(r.sum())
        if MIN_SIZE <= a < MAX_SIZE: nx += 1; out[r] = nx
    return out

def score_pair(gt, pred):
    ga = {int(i): int((gt == i).sum()) for i in np.unique(gt) if i > 0}
    pa = {int(i): int((pred == i).sum()) for i in np.unique(pred) if i > 0}
    bg = {i: 0.0 for i in ga}; bp = {j: 0.0 for j in pa}
    both = (gt > 0) & (pred > 0)
    if both.any():
        pr_, cnt = np.unique(np.stack([gt[both], pred[both]]), axis=1, return_counts=True)
        for (gi, pj), it in zip(pr_.T, cnt):
            v = it / (ga[int(gi)] + pa[int(pj)] - it)
            bg[int(gi)] = max(bg[int(gi)], v); bp[int(pj)] = max(bp[int(pj)], v)
    return bg, bp, len(ga), len(pa)

rows, tp_p, n_p = [], 0, 0
for gf in gts:
    tif = sorted((ROOT / gf.stem).glob('*.tiff'))
    im = cv2.imread(str(tif[len(tif) // 2])); gt = np.load(gf)
    bg, bp, ng, npd = score_pair(gt, seg_fix_c(im, H))
    tp_p += sum(v >= TAU for v in bp.values()); n_p += npd
    rows += [dict(slide=gf.stem, inst=i, **{'rec@0.5': int(v >= TAU)}) for i, v in bg.items()]
fc = pd.DataFrame(rows).merge(shp, on=['slide', 'inst'])
fc.to_csv('/kaggle/working/fixC_h9_percell.csv', index=False)

print(f"\nfix_C h=9, all 25 fields: cells {len(fc)}  recall {100*fc['rec@0.5'].mean():.1f}%  "
      f"per-field mean {100*fc.groupby('slide')['rec@0.5'].mean().mean():.1f}%  precision {100*tp_p/n_p:.1f}%")
print("\nTABLE 3 row - sickle-fraction error (pp)")
for thr in [0.65, 0.70, 0.75]:
    true = (fc.ecc >= thr).mean(); obs = (fc[fc['rec@0.5'] == 1].ecc >= thr).mean()
    print(f"  ecc >= {thr:.2f}  true {100*true:5.1f}%  fix_C {100*obs:5.1f}%  error {100*(obs-true):+5.2f}")
print("\ncompare released: -5.92 / -5.26 / -4.43   fix_B: -5.12 / -4.93 / -3.92")

25 gt fields from /kaggle/working/gt_masks


KeyboardInterrupt: 

In [2]:
   gt_shape: /kaggle/input/...gt_shape.csv
   25 fields
     1/25 280120-07     18s

SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers (2623762039.py, line 3)

In [3]:
import numpy as np, cv2, pandas as pd, time
from pathlib import Path
from scipy.ndimage import binary_fill_holes
from skimage.morphology import h_maxima
from skimage.measure import label as sklabel
from skimage.segmentation import watershed as skws

ROOT = Path('/kaggle/input/datasets/philominajenifer/thin-flims-scd/thin_films_part1/sickle3-edofed')
GT   = Path('/kaggle/working/gt_masks')
MIN_SIZE, MAX_SIZE, TAU, H = 5000, 17000, 0.5, 9

# ---- find gt_shape.csv quickly (searches only the small results dataset) ----
SHP = None
for p in [Path('/kaggle/working/gt_shape.csv'),
          Path('/kaggle/input/datasets/philominajenifer/paper2-gt-and-results'),
          Path('/kaggle/input/paper2-gt-and-results')]:
    if p.is_file():
        SHP = p; break
    if p.is_dir():
        hits = list(p.rglob('gt_shape.csv'))
        if hits: SHP = hits[0]; break
print('gt_shape:', SHP, flush=True)
shp = pd.read_csv(SHP)

def _front(im):
    g = cv2.GaussianBlur(cv2.cvtColor(im, cv2.COLOR_BGR2GRAY), (5, 5), 0)
    _, th = cv2.threshold(g, 127, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    k = np.ones((3, 3), np.uint8)
    op = cv2.morphologyEx(th, cv2.MORPH_OPEN, k, iterations=2)
    return op, cv2.dilate(op, k, iterations=2), cv2.distanceTransform(op, cv2.DIST_L2, 5)

def seg_fix_c(im, h):
    op, bg, dist = _front(im); fg = op > 0
    seeds = sklabel(h_maxima(dist, h) > 0)
    if seeds.max() == 0: return np.zeros(fg.shape, np.int32)
    lab = skws(-dist, seeds, mask=fg); out = np.zeros(lab.shape, np.int32); nx = 0
    for i in range(1, lab.max() + 1):
        r = binary_fill_holes(lab == i); a = int(r.sum())
        if MIN_SIZE <= a < MAX_SIZE: nx += 1; out[r] = nx
    return out

def score_pair(gt, pred):
    ga = {int(i): int((gt == i).sum()) for i in np.unique(gt) if i > 0}
    pa = {int(i): int((pred == i).sum()) for i in np.unique(pred) if i > 0}
    bg = {i: 0.0 for i in ga}; bp = {j: 0.0 for j in pa}
    both = (gt > 0) & (pred > 0)
    if both.any():
        pr_, cnt = np.unique(np.stack([gt[both], pred[both]]), axis=1, return_counts=True)
        for (gi, pj), it in zip(pr_.T, cnt):
            v = it / (ga[int(gi)] + pa[int(pj)] - it)
            bg[int(gi)] = max(bg[int(gi)], v); bp[int(pj)] = max(bp[int(pj)], v)
    return bg, bp, len(ga), len(pa)

gts = sorted(GT.glob('280120-*.npy'))
print(len(gts), 'fields', flush=True)

rows, tp_p, n_p, t0 = [], 0, 0, time.time()
for k, gf in enumerate(gts, 1):
    tif = sorted((ROOT / gf.stem).glob('*.tiff'))
    im = cv2.imread(str(tif[len(tif) // 2])); gt = np.load(gf)
    bg, bp, ng, npd = score_pair(gt, seg_fix_c(im, H))
    tp_p += sum(v >= TAU for v in bp.values()); n_p += npd
    rows += [dict(slide=gf.stem, inst=i, **{'rec@0.5': int(v >= TAU)}) for i, v in bg.items()]
    print(f'  {k:2d}/25 {gf.stem}  {time.time()-t0:5.0f}s', flush=True)

fc = pd.DataFrame(rows).merge(shp, on=['slide', 'inst'])
fc.to_csv('/kaggle/working/fixC_h9_percell.csv', index=False)

print(f"\nfix_C h=9: cells {len(fc)}  pooled recall {100*fc['rec@0.5'].mean():.1f}%  "
      f"per-field {100*fc.groupby('slide')['rec@0.5'].mean().mean():.1f}%  "
      f"precision {100*tp_p/n_p:.1f}%")
print("TABLE 3 row - sickle-fraction error (pp)")
for thr in [0.65, 0.70, 0.75]:
    true = (fc.ecc >= thr).mean()
    obs = (fc[fc['rec@0.5'] == 1].ecc >= thr).mean()
    print(f"  ecc >= {thr:.2f}  true {100*true:5.1f}%  fix_C {100*obs:5.1f}%  error {100*(obs-true):+5.2f}")

gt_shape: /kaggle/working/gt_shape.csv
25 fields
   1/25 280120-07     69s
   2/25 280120-08    114s
   3/25 280120-09    171s
   4/25 280120-10    227s
   5/25 280120-11    285s
   6/25 280120-12    341s
   7/25 280120-13    404s
   8/25 280120-14    466s
   9/25 280120-15    523s
  10/25 280120-18    571s
  11/25 280120-19    626s
  12/25 280120-20    688s
  13/25 280120-21    753s
  14/25 280120-22    812s
  15/25 280120-24    861s
  16/25 280120-26    908s
  17/25 280120-29    961s
  18/25 280120-30   1012s
  19/25 280120-31   1069s
  20/25 280120-35   1116s
  21/25 280120-36   1168s
  22/25 280120-38   1228s
  23/25 280120-39   1284s
  24/25 280120-40   1337s
  25/25 280120-41   1389s

fix_C h=9: cells 3762  pooled recall 80.0%  per-field 79.9%  precision 94.1%
TABLE 3 row - sickle-fraction error (pp)
  ecc >= 0.65  true  38.9%  fix_C  35.7%  error -3.24
  ecc >= 0.70  true  30.9%  fix_C  27.5%  error -3.37
  ecc >= 0.75  true  23.1%  fix_C  20.1%  error -3.02


In [4]:
import shutil, os
from pathlib import Path
from IPython.display import FileLink

W = Path('/kaggle/working')
OUT = W / 'paper2_results_final'
shutil.rmtree(OUT, ignore_errors=True); OUT.mkdir()

keep_ext = {'.csv', '.xlsx', '.json', '.txt', '.png', '.pdf', '.tif', '.tiff', '.eps', '.svg'}
n = 0
for p in W.rglob('*'):
    if not p.is_file() or OUT in p.parents: continue
    if p.suffix.lower() not in keep_ext: continue          # skips .npy masks, weights, zips
    if p.stat().st_size > 50e6: continue                    # skip anything huge
    rel = p.relative_to(W)
    dest = OUT / rel; dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(p, dest); n += 1
    print(f'{p.stat().st_size/1e6:7.2f} MB  {rel}')

(OUT / 'RESULTS_SUMMARY.txt').write_text("""PAPER 2 - KEY NUMBERS (25 Sep 2026)

fix_C h=9, primary cohort, all 25 fields
  cells 3762 | pooled recall 80.0% | per-field recall 79.9% | precision 94.1%
  sickle-fraction error (pp): ecc>=0.65 -3.24 | 0.70 -3.37 | 0.75 -3.02
  (released -5.92/-5.26/-4.43 ; fix_B -5.12/-4.93/-3.92 ; cellpose -1.34/-1.29/-1.34)

Batch-matched cohort (24 slides) - from bm_ridge_ratio.csv
  class gap 18.8 pp, 95% CI 5.6 to 31.5 (slide bootstrap), MW P = 0.020
  ratio-only b = 87.2 [52.9, 121.5], R2 0.558
  + class: class coef -5.9 [-18.8, 6.9], P = 0.35, R2 0.577 -> ratio explains ~68%
  fix_C h=9 gap 4.6 pp, 95% CI -6.5 to 13.9, P = 0.12

Averages: 55.8% = per-field mean, 55.7% = pooled over cells;
          43.5% = slide mean (SCD+), 43.3% = pooled over cells
""")

zip_path = shutil.make_archive(str(W / 'paper2_results_final'), 'zip', str(OUT))
print(f'\n{n} files -> {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)')
FileLink('paper2_results_final.zip')

   0.01 MB  audit_per_field.csv
   0.29 MB  fixC_h9_percell.csv
   0.01 MB  audit_trace_full.csv
   0.01 MB  field_coverage.csv
   0.28 MB  gt_shape.csv
   0.00 MB  gt_summary.csv
   1.02 MB  audit_cells_full.csv
   0.00 MB  gt_config.json
   0.01 MB  slide_labels.csv
   0.01 MB  audit_trace.csv
   0.88 MB  audit_cells.csv
   0.00 MB  unreadable_tiffs.csv
   0.03 MB  audit_precision.csv
   1.32 MB  figures/fig0_methods_overview.tiff
   0.78 MB  figures/fig0_methods_overview.png
   0.05 MB  figures/fig0_methods_overview.pdf
   0.01 MB  paper2_results/audit_per_field.csv
   0.01 MB  paper2_results/audit_trace_full.csv
   0.01 MB  paper2_results/field_coverage.csv
   0.28 MB  paper2_results/gt_shape.csv
   0.00 MB  paper2_results/gt_summary.csv
   1.02 MB  paper2_results/audit_cells_full.csv
   0.00 MB  paper2_results/gt_config.json
   0.01 MB  paper2_results/slide_labels.csv
   0.01 MB  paper2_results/audit_trace.csv
   0.88 MB  paper2_results/audit_cells.csv
   0.00 MB  paper2_results/u

/kaggle/working/paper2_results_final.zip

In [5]:
"""
Paper 2 (MBEC) - Figures 1-5 and graphical abstract, regenerated from the result tables.

Inputs (all in DATA):
  audit_cells_full.csv, gt_shape.csv, sickle_fraction_error.csv, sickle_fraction_perslide.csv,
  bm_ridge_ratio.csv, bm_alpha_summary.csv
Outputs (in OUT): FigN.pdf / .eps / .tiff (600 dpi) / .png, Graphical_abstract.*

Run locally or on Kaggle:  python paper2_figures.py  (edit DATA / OUT below)
Fig. 6 needs the TIFF fields and masks, so it is produced by the separate Kaggle cell.
"""
import os, numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from scipy import stats
from scipy.ndimage import distance_transform_edt
from skimage.measure import label as sklabel
from skimage.segmentation import watershed
from PIL import Image
import statsmodels.api as sm

DATA = os.environ.get("P2_DATA", "/kaggle/working")
OUT = os.environ.get("P2_OUT", "/kaggle/working/paper2_figures")
os.makedirs(OUT, exist_ok=True)

matplotlib.rcParams.update({
    "pdf.fonttype": 42, "ps.fonttype": 42, "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"], "font.size": 8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.linewidth": 0.8,
    "legend.frameon": False})
# Okabe-Ito
C = dict(rel="#D55E00", fa="#E69F00", fb="#0072B2", fc="#009E73", cp="#000000",
         neg="#0072B2", pos="#D55E00", grey="#999999", iso="#BBBBBB", touch="#D55E00")
MLAB = {"original": "Released", "fix_A": "fix$_A$", "fix_B": "fix$_B$",
        "fix_C": "fix$_C$", "cellpose_sam": "Cellpose-SAM"}

def save(fig, name):
    for ext in ("pdf", "eps"):
        fig.savefig(f"{OUT}/{name}.{ext}", bbox_inches="tight")
    fig.savefig(f"{OUT}/{name}.png", dpi=600, bbox_inches="tight")
    Image.open(f"{OUT}/{name}.png").convert("RGB").save(
        f"{OUT}/{name}.tiff", compression="tiff_lzw", dpi=(600, 600))
    plt.close(fig); print("wrote", name)

def panel(ax, l): ax.set_title(l, loc="left", fontweight="bold", fontsize=10)

cells = pd.read_csv(f"{DATA}/audit_cells_full.csv")
shape = pd.read_csv(f"{DATA}/gt_shape.csv")
m = cells.merge(shape, on=["slide", "inst"], suffixes=("", "_s"))

# =====================================================================
# FIG 1  rationale (a) and illustrative mechanism (b)
# =====================================================================
def box(ax, x, y, w, h, txt, fc="white", ec="#555555", fs=7, bold=False):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.03",
                                fc=fc, ec=ec, lw=0.9))
    ax.text(x + w / 2, y + h / 2, txt, ha="center", va="center", fontsize=fs,
            fontweight="bold" if bold else "normal", linespacing=1.25)

def arrow(ax, x0, y0, x1, y1):
    ax.add_patch(FancyArrowPatch((x0, y0), (x1, y1), arrowstyle="-|>", mutation_scale=9,
                                 lw=0.9, color="#333333"))

fig = plt.figure(figsize=(7.2, 5.6))
gs = fig.add_gridspec(3, 4, height_ratios=[0.62, 1, 1], hspace=0.38, wspace=0.42)
axA = fig.add_subplot(gs[0, :]); axA.set_xlim(0, 10); axA.set_ylim(0.1, 2.0); axA.axis("off")
panel(axA, "a")
xs = [0.05, 2.05, 4.35, 6.25, 8.3]; w = [1.7, 2.0, 1.6, 1.75, 1.65]
labels = ["Thin blood film\nfield", "Segmentation\n(distance transform,\nseeds > 0.3 × field max,\nwatershed, area filter)",
          "Crop & rescale\n128 × 128 px", "Classifier\nsickled / normal", "Sickle fraction\n(diagnostic output)"]
fcs = ["#F4F4F4", "#FBE3D6", "#F4F4F4", "#DCEBF7", "#F4F4F4"]
for x, ww, t, f in zip(xs, w, labels, fcs):
    box(axA, x, 0.55, ww, 1.25, t, fc=f)
for i in range(4):
    arrow(axA, xs[i] + w[i] + 0.02, 1.17, xs[i + 1] - 0.03, 1.17)
axA.text(xs[1] + w[1] / 2, 0.25, "audited here: never evaluated at instance level",
         ha="center", fontsize=6.5, color=C["rel"], style="italic")
axA.text(xs[3] + w[3] / 2, 0.25, "published: precision 0.86, recall 0.69",
         ha="center", fontsize=6.5, color=C["fb"], style="italic")

# illustrative synthetic fields
H, W = 140, 260
yy, xx = np.mgrid[0:H, 0:W]
def disk(cx, cy, r): return (xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2
def ell(cx, cy, a, b): return ((xx - cx) / a) ** 2 + ((yy - cy) / b) ** 2 <= 1
rows = [("round pair", disk(88, 70, 40) | disk(166, 70, 40)),
        ("elongated + round", ell(80, 70, 66, 22) | disk(180, 70, 40))]
FIELD_MAX = 110.0; T = 0.3 * FIELD_MAX        # illustrative field-wide maximum elsewhere in the image
CEIL = 1.6 * np.pi * 40 ** 2                  # illustrative area ceiling (1.6 × one round cell)
for r, (name, mask) in enumerate(rows, start=1):
    dt = distance_transform_edt(mask)
    seeds = sklabel(dt > T)
    lab = watershed(-dt, seeds, mask=mask)
    ax = fig.add_subplot(gs[r, 0]); ax.imshow(mask, cmap="Greys", vmin=0, vmax=1.6); ax.axis("off")
    ax.text(-8, H / 2, name, rotation=90, ha="right", va="center", fontsize=7)
    if r == 1: ax.set_title("cells", fontsize=7.5, pad=2)
    ax = fig.add_subplot(gs[r, 1]); ax.imshow(dt, cmap="magma"); ax.axis("off")
    ax.axhline(70, color="w", lw=0.6, ls="--")
    if r == 1: ax.set_title("distance transform", fontsize=7.5, pad=2)
    ax = fig.add_subplot(gs[r, 2]); ax.plot(dt[70], color="k", lw=1)
    ax.axhline(T, color=C["rel"], lw=1, ls="--"); ax.set_ylim(0, 50); ax.set_xticks([]); ax.tick_params(labelsize=6.5)
    ax.text(2, T + 1.5, "0.3 × field max", ha="left", fontsize=6, color=C["rel"])
    ax.set_ylabel("DT (px)", fontsize=7)
    if r == 1: ax.set_title("profile through centres", fontsize=7.5, pad=2)
    ax = fig.add_subplot(gs[r, 3]); ax.axis("off")
    rgb = np.ones((H, W, 3))
    n_seed = seeds.max(); regions = [lab == i for i in range(1, lab.max() + 1)]
    kept_any = False
    for reg in regions:
        a = reg.sum()
        col = (0.0, 0.62, 0.45) if a < CEIL else (0.84, 0.37, 0.0)
        rgb[reg] = col; kept_any |= a < CEIL
    rgb[seeds > 0] = (1, 1, 1)
    ax.imshow(rgb)
    fate = ("2 seeds: both cells retained" if n_seed >= 2
            else "1 seed: basins merge, region > area\nceiling, both cells deleted")
    ax.text(W / 2, H + 18, fate, ha="center", va="top", fontsize=6.5,
            color=(C["fc"] if n_seed >= 2 else C["rel"]))
    if r == 1: ax.set_title("seeds and fate", fontsize=7.5, pad=2)
fig.text(0.075, 0.655, "b", fontweight="bold", fontsize=10)
fig.text(0.99, 0.01, "Panel b is illustrative, not measured", ha="right", fontsize=6, color="grey")
save(fig, "Fig1")

# =====================================================================
# FIG 2  recall by contact
# =====================================================================
iso_touch = {}
for meth in ["original", "fix_A", "fix_B", "cellpose_sam"]:
    g = cells[cells.method == meth]
    iso_touch[meth] = (100 * g[g.contact <= 0.01]["rec@0.5"].mean(),
                       100 * g[g.contact > 0.01]["rec@0.5"].mean())
fixc_file = f"{DATA}/fixC_h9_percell.csv"
if os.path.exists(fixc_file) and "contact" in pd.read_csv(fixc_file, nrows=1).columns:
    fcx = pd.read_csv(fixc_file)
    iso_touch["fix_C"] = (100 * fcx[fcx.contact <= 0.01]["rec@0.5"].mean(),
                          100 * fcx[fcx.contact > 0.01]["rec@0.5"].mean())
else:
    iso_touch["fix_C"] = (86.8, 51.2)       # h = 9, recorded 3 Sep 2026
order = ["original", "fix_A", "fix_B", "fix_C", "cellpose_sam"]
fig, ax = plt.subplots(figsize=(4.2, 2.6))
x = np.arange(len(order)); wd = 0.36
iv = [iso_touch[k][0] for k in order]; tv = [iso_touch[k][1] for k in order]
b1 = ax.bar(x - wd / 2, iv, wd, color=C["iso"], ec="k", lw=0.5, label="Isolated (n = 3,039)")
b2 = ax.bar(x + wd / 2, tv, wd, color=C["touch"], ec="k", lw=0.5, label="Touching (n = 723)")
for bars in (b1, b2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.2, f"{b.get_height():.0f}",
                ha="center", fontsize=6.5)
ax.set_xticks(x); ax.set_xticklabels([MLAB[k] for k in order], fontsize=7.5)
ax.set_ylabel("Recall at IoU 0.5 (%)"); ax.set_ylim(0, 110); ax.legend(fontsize=7, loc="upper left", ncol=2)
save(fig, "Fig2")

# =====================================================================
# FIG 3  discarded vs retained cells (released pipeline)
# =====================================================================
o = m[(m.method == "original") & m.fate.isin(["deleted_oversize", "matched"])]
fig, axs = plt.subplots(1, 3, figsize=(6.6, 2.4))
for ax, col, lab, l in zip(axs, ["area", "ecc", "extent"],
                           ["Area (px)", "Eccentricity", "Extent"], "abc"):
    d0 = o[o.fate == "matched"][col]; d1 = o[o.fate == "deleted_oversize"][col]
    vp = ax.violinplot([d0, d1], showextrema=False, widths=0.8)
    for body, cc in zip(vp["bodies"], [C["fb"], C["rel"]]):
        body.set_facecolor(cc); body.set_edgecolor("k"); body.set_linewidth(0.4); body.set_alpha(0.6)
    for i, d in enumerate([d0, d1], 1):
        ax.hlines(np.median(d), i - 0.22, i + 0.22, color="k", lw=1.5)
    per = o.groupby(["slide", "fate"])[col].median().unstack().dropna()
    p = stats.wilcoxon(per["deleted_oversize"], per["matched"]).pvalue
    ptxt = f"$P$ = {p:.2f}" if p >= 0.01 else f"$P$ = {p:.1e}".replace("e-0", "×10$^{-").replace("e-", "×10$^{-") + "}$"
    ax.set_title(ptxt, fontsize=7.5)
    ax.set_xticks([1, 2]); ax.set_xticklabels([f"retained\n(n = {len(d0):,})", f"deleted oversize\n(n = {len(d1):,})"], fontsize=6.8)
    ax.set_ylabel(lab); panel(ax, l); ax.title.set_position((0.5, 1.0))
    if col == "area":
        ax.axhline(17000, color="grey", ls=":", lw=1); ax.text(0.45, 17400, "17,000 px ceiling", ha="left", fontsize=6, color="grey")
axs[0].set_title("a", loc="left", fontweight="bold", fontsize=10)
axs[1].set_title("b", loc="left", fontweight="bold", fontsize=10)
axs[2].set_title("c", loc="left", fontweight="bold", fontsize=10)
fig.tight_layout(w_pad=1.5)
save(fig, "Fig3")

# =====================================================================
# FIG 4  sickle-fraction error
# =====================================================================
se = pd.read_csv(f"{DATA}/sickle_fraction_error.csv")
ps = pd.read_csv(f"{DATA}/sickle_fraction_perslide.csv")
fixc_err = {0.65: -3.24, 0.70: -3.37, 0.75: -3.02}     # h = 9, run of 25 Sep 2026
fig, axs = plt.subplots(1, 2, figsize=(6.2, 2.6), gridspec_kw=dict(width_ratios=[1.25, 1]))
ax = axs[0]
for meth, cc in [("original", C["rel"]), ("fix_B", C["fb"]), ("cellpose_sam", C["cp"])]:
    g = se[se.method == meth]; ax.plot(g.thr, g.err, "-o", color=cc, ms=3.5, lw=1.2, label=MLAB[meth])
ax.plot(list(fixc_err), list(fixc_err.values()), "-o", color=C["fc"], ms=3.5, lw=1.2, label="fix$_C$ ($h$ = 9)")
ax.axhline(0, color="grey", lw=0.8); ax.set_xlabel("Eccentricity threshold for 'sickled'")
ax.set_ylabel("Error in sickle fraction (pp)"); ax.legend(fontsize=6.5, loc="upper center", ncol=2); panel(ax, "a")
ax.set_ylim(-7.2, 1.9)
ax = axs[1]
ax.scatter(ps.true, ps.rep, s=14, color=C["rel"], zorder=3)
lim = [10, 55]; ax.plot(lim, lim, "k--", lw=0.8); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("True sickle fraction (%)"); ax.set_ylabel("Reported, released pipeline (%)")
ax.text(12, 51, f"{(ps.d < 0).sum()} of {len(ps)} slides under-reported\nmean {ps.d.mean():+.1f} pp", fontsize=6.5)
panel(ax, "b"); fig.tight_layout(w_pad=2)
save(fig, "Fig4")

# =====================================================================
# FIG 5  phenotype-aligned disparity and mechanism
# =====================================================================
d = pd.read_csv(f"{DATA}/bm_ridge_ratio.csv"); a = pd.read_csv(f"{DATA}/bm_alpha_summary.csv")
fig, ax = plt.subplots(1, 4, figsize=(7.5, 2.35), gridspec_kw=dict(width_ratios=[1, 1.1, 1.1, 0.9]))
rng = np.random.default_rng(3)
for xpos, (k, cc) in enumerate([(0, C["neg"]), (1, C["pos"])]):
    y = d[d.pos == k].recall
    ax[0].scatter(xpos + rng.uniform(-.12, .12, len(y)), y, s=16, color=cc, zorder=3)
    ax[0].hlines(y.mean(), xpos - .22, xpos + .22, color=cc, lw=2)
ax[0].set_xticks([0, 1]); ax[0].set_xticklabels(["SCD−\n(n = 9)", "SCD+\n(n = 15)"], fontsize=7)
ax[0].set_xlim(-.5, 1.5); ax[0].set_ylabel("Per-slide recall at IoU 0.5 (%)")
ax[0].text(.35, 79, "$P$ = 0.020", fontsize=7)
fit = sm.OLS(d.recall, sm.add_constant(d.ratio)).fit(); xx_ = np.linspace(.25, .82, 50)
ax[1].plot(xx_, fit.params.iloc[0] + fit.params.iloc[1] * xx_, "k", lw=1)
for k, cc, l in [(0, C["neg"], "SCD−"), (1, C["pos"], "SCD+")]:
    s = d[d.pos == k]; ax[1].scatter(s.ratio, s.recall, s=12, color=cc, label=l, zorder=3)
ax[1].axvline(.30, color="grey", ls=":", lw=1); ax[1].text(.31, 15.5, "seed thr.", fontsize=6, color="grey")
ax[1].text(.26, 73, f"$R^2$ = {fit.rsquared:.2f}\nclass $P$ = 0.35", fontsize=7)
ax[1].set_xlabel("Ridge / field max"); ax[1].set_ylabel("Per-slide recall (%)"); ax[1].legend(fontsize=6.5, loc="lower right")
ax[2].plot(a.alpha, a.neg, "-o", color=C["neg"], ms=3.5, label="SCD−")
ax[2].plot(a.alpha, a.pos, "-o", color=C["pos"], ms=3.5, label="SCD+")
ax[2].axvline(.30, color="grey", ls=":", lw=1); ax[2].text(.305, 21, "released", fontsize=6, color="grey")
ax[2].set_xlabel("Seed coefficient $\\alpha$"); ax[2].set_ylabel("Recall (%)"); ax[2].legend(fontsize=6.5, loc="lower left")
vals, pv = [18.8, 5.4, 4.6], ["0.020", "0.37", "0.12"]
ax[3].bar([0, 1, 2], vals, color=[C["rel"], C["fb"], C["fc"]], width=.65)
for i, (v, p) in enumerate(zip(vals, pv)): ax[3].text(i, v + .4, f"{v}\n$P$ = {p}", ha="center", fontsize=6.5)
ax[3].set_xticks([0, 1, 2]); ax[3].set_xticklabels(["Released", "fix$_B$", "fix$_C$"], fontsize=7)
ax[3].set_ylim(0, 24); ax[3].set_ylabel("SCD− minus SCD+ (pp)")
for i, l in enumerate("abcd"): panel(ax[i], l)
fig.tight_layout(w_pad=1.2)
save(fig, "Fig5")

# =====================================================================
# GRAPHICAL ABSTRACT  (aspect 32.93 : 37.63, width : height; vector PDF scales freely)
# =====================================================================
fig = plt.figure(figsize=(3.293 * 1.6, 3.763 * 1.6))
ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, 10); ax.set_ylim(0, 11.43); ax.axis("off")
ax.text(5, 11.02, "Segmentation errors that track the disease", ha="center", fontsize=10.5, fontweight="bold")
def gbox(y, h, title, fc):
    ax.add_patch(FancyBboxPatch((0.3, y), 9.4, h, boxstyle="round,pad=0.02,rounding_size=0.15", fc=fc, ec="#777777", lw=0.8))
    ax.text(0.6, y + h - 0.28, title, fontsize=8.5, fontweight="bold", va="top")
def ins(x, y, w, h): return ax.inset_axes([x, y, w, h], transform=ax.transData)
# 1 -- pipeline
gbox(8.25, 2.4, "1  Segmentation is the unevaluated first stage", "#F4F4F4")
for k, (xx, t, fc) in enumerate([(0.7, "blood film\nfield", "white"), (3.55, "segmentation", "#FBE3D6"),
                                 (6.4, "classifier", "#DCEBF7")]):
    ax.add_patch(FancyBboxPatch((xx, 8.75), 2.3, 1.05, boxstyle="round,pad=0.02,rounding_size=0.1", fc=fc, ec="#555555", lw=0.7))
    ax.text(xx + 1.15, 9.27, t, ha="center", va="center", fontsize=7.5)
    if k < 2: ax.add_patch(FancyArrowPatch((xx + 2.33, 9.27), (xx + 2.82, 9.27), arrowstyle="-|>", mutation_scale=8, color="#333333"))
ax.text(4.7, 8.45, "cells it misses never reach the classifier", ha="center", fontsize=6.8, style="italic", color=C["rel"])
# 2 -- disparity
gbox(4.45, 3.55, "2  It loses more cells from sickle cell films", "#FBE3D6")
b = ins(1.2, 5.25, 4.0, 2.0)
b.bar([0, 1], [62.3, 43.5], color=[C["neg"], C["pos"]], width=0.6)
for i2, v in enumerate([62.3, 43.5]): b.text(i2, v + 3, f"{v}%", ha="center", fontsize=7.5, fontweight="bold")
b.set_xticks([0, 1]); b.set_xticklabels(["control", "sickle cell"], fontsize=7); b.set_ylim(0, 82)
b.set_yticks([]); b.spines["left"].set_visible(False); b.patch.set_alpha(0)
b.set_title("cells recovered", fontsize=7, pad=1)
ax.text(5.6, 6.75, "Cause:", fontsize=7.5, fontweight="bold")
ax.text(5.6, 6.35, "seed threshold scaled\nto the largest structure\nin the whole field", fontsize=7.2, va="top", linespacing=1.3)
ax.text(5.6, 5.2, "explains ~2/3 of the gap", fontsize=7, style="italic")
# 3 -- consequence and repair
gbox(0.3, 3.85, "3  Consequence and repair", "#DDF1EA")
c1 = ins(1.0, 1.35, 3.4, 1.85)
c1.bar([0, 1], [30.9, 25.6], color=[C["grey"], C["rel"]], width=0.6)
for i2, v in enumerate([30.9, 25.6]): c1.text(i2, v + 1.5, f"{v}%", ha="center", fontsize=7.5, fontweight="bold")
c1.set_xticks([0, 1]); c1.set_xticklabels(["true", "reported"], fontsize=7); c1.set_ylim(0, 40)
c1.set_yticks([]); c1.spines["left"].set_visible(False); c1.patch.set_alpha(0)
c1.set_title("sickle fraction: 17% under-count", fontsize=7, pad=1)
c2 = ins(5.6, 1.35, 3.4, 1.85)
c2.bar([0, 1], [18.8, 4.6], color=[C["rel"], C["fc"]], width=0.6)
for i2, v in enumerate([18.8, 4.6]): c2.text(i2, v + 0.8, f"{v}", ha="center", fontsize=7.5, fontweight="bold")
c2.set_xticks([0, 1]); c2.set_xticklabels(["released", "fixed seeding"], fontsize=7); c2.set_ylim(0, 24)
c2.set_yticks([]); c2.spines["left"].set_visible(False); c2.patch.set_alpha(0)
c2.set_title("class gap (points)", fontsize=7, pad=1)
ax.text(5, 0.5, "Evaluate segmentation for attribute-independent error, not aggregate recall",
        ha="center", fontsize=6.6, style="italic")
for y0, y1 in [(8.22, 8.03), (4.42, 4.2)]:
    ax.add_patch(FancyArrowPatch((5, y0), (5, y1), arrowstyle="-|>", mutation_scale=10, color="#333333"))
save(fig, "Graphical_abstract")
print("done ->", OUT)

wrote Fig1
wrote Fig2


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


wrote Fig3


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/sickle_fraction_error.csv'